# Dialog servers and browser ES modules
> Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.

In [ ]:
#| default_exp server

In [ ]:
#| export
#| hide
import asyncio, inspect, json, keyword, logging, os, subprocess, re
from contextlib import asynccontextmanager
from collections.abc import Callable, AsyncIterator
from datetime import datetime
from functools import partial
from urllib.parse import quote
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
from starlette.responses import Response
from starlette.routing import Route
from fastcore.basics import first
from fastcore.aio import maybe_await
from fastcore.meta import delegates
from fastcore.net import is_port_free
from fastcore.xml import FT
from fasthtml.core import FastHTML
from fasthtml.jupyter import JupyUvi, JupyUviAsync, HTMX
from fasthtml.xtend import Script, Link
from dialoghelper.core import js_eval_a, read_msg
from pote.basic import Kounter
from pote.dutil import link_msg, set_var

In [ ]:
#| hide
import socket
from multiprocessing import Queue
import fastcore.docments
from itertools import count
from unittest.mock import patch as mock_patch
from starlette.testclient import TestClient
from ipykernel_helper import info_md
from fastcore.basics import setify, ifnone
from fastcore.foundation import store_attr, patch
from fastcore.parallel import startproc
from fastcore.test import *
from fasthtml.common import fast_app
from pyskills import doc
from pyskills import *
from fasthtml.components import Div, Span, P
from dialoghelper.core import add_msg, iife_a, add_html_a, add_scr_a, js_run_a, del_msgs
from htmx_bridge import get_hx
from pote.logger import logger
from pote.dutil import wait_for

In [ ]:
#| exporti
kntr = Kounter()

In [ ]:
hx = get_hx()

In [ ]:
try:
    if globals()['srv']: await maybe_await(srv.stop())
except BaseException: pass
srv=app=rt=p = None

### Choosing a helper

```python
from pote.server import ensure_server_a, serve_app, setup_mod, add_js_msg
```

- `await ensure_server_a(...)` starts or reuses a FastHTML server and returns `(srv, app, rt, p)`; by default it selects a free non-public port.
- `async with serve_app(...)` is for a scoped server that is stopped automatically, particularly in tests.
- `find_server`, `is_serving_a`, and `stop_server_a` inspect and clean up existing server state.
- `free_port(public=True)` returns first available public port.
- `p(path)` previews a route in an iframe on a public port; it is None for private ports.
- `setup_mod(..., versioned=True)` supports iterative browser ES-module work.
- `get_js`, `add_js_msg`, and `setup_mod` without `src` extract JavaScript from a `%%js`/`%%javascript` message.

The async APIs are normally preferable in Solveit and Jupyter. These helpers target FastHTML applications; the underlying `JupyUvi` can serve more general ASGI applications, but `pote.server` does not promise that compatibility.

## references

In [ ]:
# from inspect import getsource
# import fasthtml.jupyter
# await add_msg(getsource(fasthtml.jupyter))

`fasthtml.jupyter` provides utilities for running and displaying FastHTML/HTMX web apps **inside notebooks** (Jupyter, Solveit, etc.). Here's what the key pieces do:

**Server management**
- `nb_serve` / `nb_serve_async` — start a uvicorn server in a background thread (or async task) so it doesn't block the kernel. Waits until the server is actually ready before returning.
- `JupyUvi` / `JupyUviAsync` — convenient wrapper classes around the above, with `.start()` / `.stop()` lifecycle methods and automatic HTMX port configuration when not in Solveit.
- `wait_port_free` — blocks until a port is released (useful after `.stop()`).

**Display helpers**
- `show` — renders FastHTML components in the notebook, and triggers `htmx.process()` so htmx attributes on dynamically injected HTML are activated.
- `render_ft` — patches `FT._repr_html_` so FastHTML components auto-render with HTMX support when returned from a cell.
- `HTMX` — displays a running app in an **iframe** inside the notebook, either by URL or by rendering an `FT` component directly via a temp route.
- `htmx_config_port` — injects a JS snippet that rewrites relative htmx request paths to include the correct host/port, so htmx calls from the iframe hit the right server.

**WebSocket helper**
- `ws_client` — sets up a WebSocket-connected component displayed in an iframe, with a `send()` helper for pushing updates from Python to the browser.

In short: it bridges the gap between a notebook kernel and a live FastHTML web app, handling server lifecycle, port routing, and in-notebook rendering.

Does JupyUvi need a FastHTML app or any Starlette app will do?

---

`JupyUvi` only needs an **ASGI app**, so a plain Starlette app will do.

The constructor passes `app` straight into:

```python
uvicorn.Config(app, log_level=log_level, host=host, port=port, **kwargs)
```

so uvicorn just needs something ASGI-callable.

The only FastHTML-specific-ish part is optional `live=True`:

```python
if live: self._setup_live(app)
```

`_setup_live` assumes the app has FastHTML-style conveniences:

```python
app.hdrs.append(...)
@app.get(rt)
async def _sse(): ...
```

So:

- `JupyUvi(app)` works with any ASGI app Starlette/ FastAPI/FastHTML, etc.
- `JupyUvi(app, live=True)` expects an app with `.hdrs` and `.get`, so that's really for FastHTML-style apps unless your Starlette app provides compatible attributes.
- `HTMX(..., path=FT_or_tuple, app=app)` also assumes `app.get(route)(...)`, so that convenience path is FastHTML/Starlette-route-decorator compatible.

For a normal Starlette app:

```python
from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.routing import Route

async def home(request): return PlainTextResponse('hi')

app = Starlette(routes=[Route('/', home)])
srv = JupyUvi(app, port=8000)
```

That should work fine.

In [ ]:
doc(fastcore.docments)

# module fastcore.docments:

"""Document parameters using comments.

A *docment* documents a parameter with a trailing comment on its line in the signature, right next to the name, type, and default it describes; `docments(f)` retrieves them programmatically, so tools (docs tables, CLIs like `fastcore.script`, ...) can use them. This replaces repeating parameter names in a specially-formatted docstring, which has to be kept in sync with the signature by hand:

```python
def add(
    a:int,  # The first operand
    b=1,    # The second operand
) -> int:   # The sum
    "Add `a` to `b`"
    return a + b
test_eq(docments(add), {'a': 'The first operand', 'b': 'The second operand', 'return': 'The sum'})
```

Comment lines *above* a param also work when one line isn't enough; `typing.Annotated` string metadata is an alternative to comments; numpy-style docstring sections are merged in when present; and `docments(f, full=True)` returns each param's default, type, and docment together.

Docs: 

## helpers

In [ ]:
#| exporti
def kill_port(port):
    "Forcibly terminate local processes listening on `port`; intended only for notebook recovery"
    subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

In [ ]:
def listen(q):
    s = socket.socket(); s.bind(('', 0)); s.listen()
    q.put(s.getsockname()[1]); s.accept()

q = Queue()
proc = startproc(lambda: listen(q), daemon=True)
port = q.get(timeout=5)
kill_port(port)
proc.join(5)

test_eq(proc.is_alive(), False)
test_is(is_port_free(port), True)

In [ ]:
#| export
def wrap_endp(
    response, # Existing request handler or fixed response value
) -> Callable:
    "Return callable handlers unchanged; otherwise return an async handler that returns `response` unchanged."
    if callable(response) and not isinstance(response, FT) and not hasattr(response, '__ft__'): return response
    async def endpoint(request): return response
    return endpoint

In [ ]:
fixed = 'ok'
ep = wrap_endp(fixed)
test_eq(await ep(None), fixed)

ft = Div('x')
test_is(await wrap_endp(ft)(None), ft)

seq = (Div('a'), Span('b'))
test_is(await wrap_endp(seq)(None), seq)

sync_handler = lambda request: 'sync'
test_is(wrap_endp(sync_handler), sync_handler)

async def async_handler(request): return 'async'
test_is(wrap_endp(async_handler), async_handler)

component = lambda: None
component.__ft__ = lambda: Div('x')
test_is(await wrap_endp(component)(None), component)

In [ ]:
json.loads(os.environ["PUBLIC_DOMAINS"]).keys()

dict_keys(['6000', '8000', '8001', '8002', '8003', '8004', '8005', '8006', '8007'])

In [ ]:
#| export
def pub_ports() -> tuple[int]:
    "Public ports reserved by this Solveit instance."
    return tuple(sorted(map(int, json.loads(os.environ.get('PUBLIC_DOMAINS', '{}')))))

In [ ]:
#| export
def free_port(
    start:int=6550, # First port to consider
    skip=(), # Ports to exclude even when available
    public:bool=False, # Restrict search to this instance's public ports?
    host:str='127.0.0.1', # Host interface used to test availability
) -> int:
    "Return the first available port; raise `RuntimeError` when none are available."
    skip, ports = set(skip), pub_ports() if public else range(start, 65536)
    if (port := first(p for p in ports if p not in skip and is_port_free(p, host))) is None: raise RuntimeError("No free port")
    return port

In [ ]:
start = free_port()
next_free = free_port(start=start+1)

test_eq(free_port(start=start), start)
test_eq(free_port(start=start, skip=[start]), next_free)
test_is(free_port(public=True) in pub_ports(), True)

with mock_patch.dict(free_port.__globals__, is_port_free=lambda *args: False):
    with expect_fail(RuntimeError, 'No free port'): free_port(public=True)

free_port(public=True)

8000

In [ ]:
#| export
def free_public_domain():
    "Public domain for the first public port that is currently free."
    return json.loads(os.environ["PUBLIC_DOMAINS"])[str(free_port(public=True))]

def pub_domain(port):
    "Public domain for `port`."
    return json.loads(os.environ['PUBLIC_DOMAINS'])[str(port)]

In [ ]:
with mock_patch.dict(free_public_domain.__globals__, free_port=lambda public: 8000):
    test_eq(free_public_domain(), json.loads(os.environ["PUBLIC_DOMAINS"])['8000'])

In [ ]:
#| export
def find_server(
    port:int|None=None, # Port to match; `None` matches any port
    all_ports:bool=False, # Return every started binding across all ports; aliases may repeat the same server
) -> tuple[JupyUvi,str]|tuple[None,None]|list[tuple[JupyUvi,str]]:
    "Find namespace-bound `JupyUvi` objects whose flags indicate they are started; no HTTP health check. First-match order follows the IPython namespace."
    import IPython
    if all_ports: port = None
    ns = IPython.get_ipython().user_ns
    res = [(v,k) for k,v in ns.items() if isinstance(v, JupyUvi) and (port is None or v.port == port) and v.server and v.server.started and not v.server.should_exit]
    return res if all_ports else res[0] if res else (None,None)

In [ ]:
srv = JupyUvi(app := FastHTML(), port=free_port())

In [ ]:
srv2,_ = find_server(srv.port)
test_is(srv2, srv)

In [ ]:
test_is(find_server(srv.port)[0], srv)
test_eq(find_server()[0], srv2)
test_eq(find_server(None, all_ports=True), find_server(all_ports=True))
test_eq(set([_[1] for _ in find_server(all_ports=True)]), {'srv', 'srv2'})

In [ ]:
srv.stop()
await asyncio.sleep(0.2)
test_eq(find_server(None, all_ports=True), [])

In [ ]:
test_eq(find_server(), (None,None))
test_eq(find_server(None, all_ports=True), [])

In [ ]:
#| export
def _conn_host(host): return '127.0.0.1' if host in ('0.0.0.0', '::') else host

def is_serving(srv:JupyUvi|None, timeout=1) -> bool:
    "Return whether `srv` appears started and `GET /` completes with a status below 500."
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        return httpx.get(f"http://{_conn_host(srv.host)}:{srv.port}/", timeout=timeout).status_code < 500
    except Exception: return False

async def is_serving_a(srv:JupyUvi|None, timeout=1) -> bool:
    "Return whether `srv` appears started and `GET /` completes with a status below 500."
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        async with httpx.AsyncClient(timeout=timeout) as cli: return (await cli.get(f"http://{_conn_host(srv.host)}:{srv.port}/")).status_code < 500
    except Exception: return False

In [ ]:
test_eq(is_serving(None), False)
test_eq(await is_serving_a(None), False)

In [ ]:
p = free_port(6551, skip=pub_ports())
srv = JupyUvi(FastHTML(), port=p)
test_eq(is_serving(srv), True)
test_eq(await is_serving_a(srv), True)
srv.stop()
await asyncio.sleep(0.2)
test_eq(is_serving(srv), False)
test_eq(await is_serving_a(srv), False)

# HTTP-500 health
p = free_port(6551, skip=pub_ports())
app = FastHTML()
@app.get('/')
def unhealthy(): return Response(status_code=500)
srv = JupyUvi(app, port=p)
test_eq((is_serving(srv), await is_serving_a(srv)), (False,False))
srv.stop()
await asyncio.sleep(0.2)

In [ ]:
#| exporti
daisyui_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
)
threejs_hdrs = (
    Script('{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.165.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.165.0/examples/jsm/"}}', type='importmap'),
)

In [ ]:
def mk_app(app_kw=None):
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return FastHTML(**app_kw)

In [ ]:
a = mk_app({})
test_eq(all(h not in a.hdrs for h in daisyui_hdrs), True)

## server

In [ ]:
#| export
def _found_srvs(port=None):
    res = find_server(all_ports=True)
    if port is not None: res = [(srv,nm) for srv,nm in res if srv.port == port]
    srvs = {}
    for srv,nm in res:
        srvs.setdefault(id(srv), (srv, []))[1].append(nm)
    return list(srvs.values())

def stop_server(port:int|None=None) -> None:
    "Stop the discovered synchronous server on `port`, or all discovered servers when `port=None`."
    srvs = _found_srvs(port)
    if any(inspect.iscoroutinefunction(srv.stop) for srv,_ in srvs): raise RuntimeError("JupyUviAsync requires `await stop_server_a(port)`")
    for srv,nms in srvs:
        srv.stop()
        for nm in nms: set_var(nm, None)

async def stop_server_a(port:int|None=None) -> None:
    "Stop the discovered server on `port`, or all discovered servers when `port=None`."
    for srv,nms in _found_srvs(port):
        await maybe_await(srv.stop())
        for nm in nms: set_var(nm, None)

In [ ]:
p = free_port(6551, skip=pub_ports())
tst_srv = JupyUvi(mk_app({}), port=p)
test_is(find_server(p)[0], tst_srv)
test_eq(is_serving(tst_srv), True)
stop_server(p)
await asyncio.sleep(0.2)
test_eq(find_server(p), (None,None))
test_eq(tst_srv, None)

In [ ]:
tst_srv_a = JupyUviAsync(mk_app({}), port=p)
await tst_srv_a.start()
test_is(find_server(p)[0], tst_srv_a)
test_eq(await is_serving_a(tst_srv_a), True)

with expect_fail(RuntimeError, 'stop_server_a'): stop_server(p)

test_is(find_server(p)[0], tst_srv_a)
test_eq(await is_serving_a(tst_srv_a), True)

await stop_server_a(p)
test_eq(find_server(p), (None,None))
test_eq(tst_srv_a, None)

In [ ]:
stopped,cleared = [],[]
class TestSrv:
    def stop(self): stopped.append(self)

a,b = TestSrv(),TestSrv()
srvs = [(a, ['a', 'a_alias']), (b, ['b'])]

with mock_patch.dict(stop_server.__globals__, _found_srvs=lambda port: srvs, set_var=lambda nm,v: cleared.append((nm,v))):
    stop_server()

test_eq(stopped, [a,b])
test_eq(cleared, [('a', None), ('a_alias', None), ('b', None)])

In [ ]:
#| exporti
def _app_kw(app_kw, public):
    app_kw = dict(ifnone(app_kw, {}))
    if public: app_kw['middleware'] = [*app_kw.get('middleware', []), Middleware(
        CORSMiddleware,
        allow_origin_regex=r'''https://[a-f0-9]+\.solve\.it\.com''',
        allow_methods=['*'],
        allow_headers=['*'],
    )]
    return app_kw

def get_preview(app, port=8000, host=None):
    "Return a function that displays the app in an iframe."
    if host is None: host = None if port==8000 else json.loads(os.environ['PUBLIC_DOMAINS']).get(str(port), '127.0.0.1')
    return partial(HTMX, app=app, host=host, port=port if port in (8000, None) else 443)

def srv_bundle(srv):
    if hasattr(srv.app, 'route'): return srv,srv.app,srv.app.route,get_preview(srv.app, srv.port) if srv.port in pub_ports() else None
    else: return srv,srv.app,None,None

def _server_target(srv=None, port:int|None=None, host=None, reuse=True, public=False):
    own = False
    if srv is not None: return srv,port,host,own
    if port in pub_ports(): public = True
    if public: port,host = port if port in pub_ports() else free_port(public=True, host='0.0.0.0'),'0.0.0.0'
    check_host = host or ('0.0.0.0' if port in pub_ports() else '127.0.0.1')
    if port is None: port = free_port(6551, skip=pub_ports(), host=check_host)
    found,_ = find_server(port)
    if found:
        if public and found.host not in ('0.0.0.0', '::'): raise RuntimeError(f"Server on port {port} is not publicly bound")
        if not reuse: raise RuntimeError(f"Server already running on port {port}")
        return found,port,host,own
    if not is_port_free(port, check_host): raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
    return None,port,host,True

async def _ensure_server_a(srv=None, port:int|None=None, app=None, app_kw=None, reuse=True, public=False, **jupyuvi_kw):
    srv,port,host,own = _server_target(srv, port, jupyuvi_kw.get('host'), reuse, public)
    if srv:
        if not await is_serving_a(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return srv,own
    if host is not None: jupyuvi_kw['host'] = host
    srv = JupyUviAsync(app if app is not None else mk_app(_app_kw(app_kw, public)), port=port, **jupyuvi_kw)
    await srv.start()
    return srv,own

def _ensure_server(srv=None, port:int|None=None, app=None, app_kw=None, reuse=True, public=False, **jupyuvi_kw):
    srv,port,host,own = _server_target(srv, port, jupyuvi_kw.get('host'), reuse, public)
    if srv:
        if not is_serving(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return srv,own
    if host is not None: jupyuvi_kw['host'] = host
    srv = JupyUvi(app if app is not None else mk_app(_app_kw(app_kw, public)), port=port, **jupyuvi_kw)
    return srv,own

In [ ]:
with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     free_port=lambda **kwargs: 8000,
                     find_server=lambda port: (None,None),
                     is_port_free=lambda *args: True):
    srv,port,host,own = _server_target(port=6551, public=True)
    test_eq((srv,port,host,own), (None,8000,'0.0.0.0',True))

with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     find_server=lambda port: (None,None),
                     is_port_free=lambda *args: True):
    _,port,host,own = _server_target(port=8000, host='127.0.0.1', public=True)
    test_eq((port,host,own), (8000,'0.0.0.0',True))

with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     free_port=lambda **kwargs: 8000,
                     find_server=lambda port: (None,None),
                     is_port_free=lambda port,host: test_eq(host, '0.0.0.0') is None):
    srv,port,host,own = _server_target(public=True)
    test_eq((port,host,own), (8000,'0.0.0.0',True))

with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     find_server=lambda port: (None,None),
                     is_port_free=lambda *args: True):
    _,port,host,own = _server_target(port=8000, host='127.0.0.1')
    test_eq((port,host,own), (8000,'0.0.0.0',True))

In [ ]:
srv, own = _ensure_server()
test_is(srv is not None, True)
test_is(own, True)

srv2, _ = await _ensure_server_a(port=srv.port)
test_is(srv, srv2)

In [ ]:
_srv, sym = find_server()
test_eq((_srv, sym), (srv, 'srv'))

In [ ]:
await stop_server_a()

test_is(globals()['srv'], None)
test_is(globals()['srv2'], None)
test_eq(find_server(), (None, None))
test_is(is_serving(srv2), False)

Test explicit-port start/reuse/stop.

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,_ = _ensure_server(port=port)
test_eq((srv.port, is_serving(srv)), (port, True))

srv2,_ = _ensure_server(port=port)
test_is(srv2, srv)

await stop_server_a(port)


`port=None` chooses a non-public free port and starts cleanly:

In [ ]:
srv,_ = _ensure_server()
test_eq(srv.port in pub_ports(), False)
test_eq(srv.port >= 6551, True)

await stop_server_a(srv.port)

Passing the same app is accepted and reused:

In [ ]:
port = free_port(skip=pub_ports())
app = FastHTML()
srv1,_ = _ensure_server(port=port, app=app)
srv2,_ = _ensure_server(port=port, app=app)
test_is(srv1.app, srv2.app)

await stop_server_a(port)

In [ ]:
dead,_ = _ensure_server(port=None)
dead_ref = [dead]

dead.stop()
await asyncio.sleep(0.2)

test_eq(is_serving(dead_ref[0]), False)
test_fail(lambda: _ensure_server(srv=dead_ref[0]), contains='not serving', exc=RuntimeError)

dead_ref.clear()

Test the “existing server wins” contract explicitly, including that a passed `app` is ignored when a server already exists on the requested port.

In [ ]:
port = free_port(skip=pub_ports())
app1,app2 = FastHTML(),FastHTML()
srv,_ = _ensure_server(port=port, app=app1)
srv2,_ = await _ensure_server_a(port=port, app=app2)

test_is(srv2, srv)
test_is(srv2.app, app1)

await stop_server_a(port)

In [ ]:
srv,own = _ensure_server()
srv2,own2 = await _ensure_server_a(port=srv.port)

test_eq((own, own2), (True, False))
await stop_server_a(srv.port)

In [ ]:
port = free_port(skip=pub_ports())
srv,_ = await _ensure_server_a(port=port)

with expect_fail(RuntimeError, 'already running'):
    await _ensure_server_a(port=port, reuse=False)
test_eq(await is_serving_a(srv), True)

await stop_server_a(port)

In [ ]:
port = free_port(skip=pub_ports())
srv,_ = await _ensure_server_a(port=port)
ref = [srv]
await stop_server_a(port)

with expect_fail(RuntimeError, 'not serving'):
    await _ensure_server_a(srv=ref[0])

test_eq(await is_serving_a(ref[0]), False)
ref.clear()

In [ ]:
async def ensure_server_a(srv=None, port:int|None=8000, app=None, app_kw=None, host='127.0.0.1', public=False, **jupyuvi_kw):
    "Return a live async server, served app, route decorator, and preview helper; existing servers win"
    srv,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, public=public, **jupyuvi_kw)
    return (*srv_bundle(srv),)

def ensure_server(srv=None, port:int|None=8000, app=None, app_kw=None, host='127.0.0.1', public=False, **jupyuvi_kw):
    "Return a live server, served app, route decorator, and preview helper; existing servers win"
    srv,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, public=public, **jupyuvi_kw)
    return (*srv_bundle(srv),)

In [ ]:
# cors = [Middleware(
#     CORSMiddleware,
#     allow_origin_regex=r'''https://[a-f0-9]+\.solve\.it\.com''',
#     allow_methods=['*'],
#     allow_headers=['*'],
# )]

In [ ]:
mid1 = (await read_msg(0)).id

port = free_port(skip=(6000,), public=True)
domain = json.loads(os.environ["PUBLIC_DOMAINS"])[str(port)]

# srv, app, rt, p = await ensure_server_a(app_kw=AD(exts='ws', hdrs=None, middleware=cors), port=port, host='0.0.0.0')
srv, app, rt, p = await ensure_server_a(port=port, public=True)
test_eq((srv.port, await is_serving_a(srv)), (port, True))

@rt
def hello(): return f"Running on port {port}"

p(hello)

In [ ]:
mid2 = (await read_msg(0)).id

url = '/hello' if port == 8000 else f"https://{domain}/hello"
(c := P(hx_get=url, hx_trigger='load once', id='hello01'))

Browser console:
```
dialog_?name=prj%2Fpote%2Fnbs%2F02_server&noscroll=:1 Access to XMLHttpRequest at 'https://zen-koala-sings-pww6ud.solveit.pub/hello?dlg_name=prj%2Fpote%2Fnbs%2F02_server' from origin 'https://857cd595cfff6e6f0bdaf7999cf5ef6b.solve.it.com' has been blocked by CORS policy: Response to preflight request doesn't pass access control check: No 'Access-Control-Allow-Origin' header is present on the requested resource.

htmx-2.0.8.js:3090 htmx:afterRequest, [object HTMLParagraphElement]
logError @ htmx-2.0.8.js:3090
...
(anonymous) @ ws-2.0.3.js:147

htmx-2.0.8.js:3090 htmx:sendError, [object HTMLParagraphElement]
logError @ htmx-2.0.8.js:3090
...
(anonymous) @ ws-2.0.3.js:147

htmx-2.0.8.js:4647  GET https://zen-koala-sings-pww6ud.solveit.pub/hello?dlg_name=prj%2Fpote%2Fnbs%2F02_server net::ERR_FAILED
issueAjaxRequest @ htmx-2.0.8.js:4647
...
(anonymous) @ ws-2.0.3.js:147
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`p(hello)` works because it is an iframe navigation. The `hx-get` runs from the Solveit page itself, whose origin is:

```text
https://857cd595cfff6e6f0bdaf7999cf5ef6b.solve.it.com
```

It therefore makes a cross-origin XHR to `https://zen-koala-sings-pww6ud.solveit.pub`. HTMX adds request headers, which triggers an `OPTIONS` preflight; the FastHTML app does not currently return CORS headers.

The server must be constructed with CORS middleware. Adding it after `ensure_server_a` has started the app is too late, so the small change belongs in the CRAFT server initialization:

```python
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
```

```python
cors = [Middleware(
    CORSMiddleware,
    allow_origin_regex=r'''https://[a-f0-9]+\.solve\.it\.com''',
    allow_methods=['*'],
    allow_headers=['*'],
)]

srv, app, rt, _ = await ensure_server_a(app_kw=AD(exts='ws', hdrs=None, middleware=cors))
```

The regular expression permits Solveit dialog origins while not opening the public endpoint to arbitrary browser origins. After changing this initialization, restart the existing server so its middleware stack is rebuilt.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52137, "completion_tokens": 1846, "total_tokens": 53983, "cached_tokens": 8448, "reasoning_tokens": 1552, "cost": 0.00920505}
```


In [ ]:
await wait_for(lambda: hx.eval(f'''$('#{c}').text()''')==f"Running on port {port}")

That worked. We should incorporate that into `ensure_server_a` (when creating the app and for public ports; for private ports isn't necessary), right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. The rule should be: **when `ensure_server_a` creates a new public FastHTML app, add Solveit-origin CORS middleware automatically.** Private servers do not need it, and a caller-supplied `app` should remain untouched.

Use the explicit `public=True` contract rather than inferring from the port: public ports may still be deliberately bound privately.

Add these imports:

```python
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
```

Then add a small helper which preserves any caller-provided middleware:

```python
def _app_kw(app_kw, public):
    app_kw = dict(ifnone(app_kw, {}))
    if public: app_kw['middleware'] = [*app_kw.get('middleware', []), Middleware(
        CORSMiddleware,
        allow_origin_regex=r'''https://[a-f0-9]+\.solve\.it\.com''',
        allow_methods=['*'],
        allow_headers=['*'],
    )]
    return app_kw
```

Finally, change the new-app branch in `_ensure_server_a`:

```python
srv = JupyUviAsync(app if app is not None else mk_app(_app_kw(app_kw, public)), port=port, **jupyuvi_kw)
```

Make the identical change in `_ensure_server` so synchronous and asynchronous helpers keep the same contract. This puts CORS in place before FastHTML constructs its middleware stack, preserves user middleware, and leaves reused or explicitly supplied apps alone.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52459, "completion_tokens": 835, "total_tokens": 53294, "cached_tokens": 8448, "reasoning_tokens": 516, "cost": 0.008495100000000002}
```


In [ ]:
await update_msg(mid1, output='')
await update_msg(mid2, output='')
await stop_server_a()

In [ ]:
try:
    port = free_port(6551, skip=pub_ports())
    srv,app,rt,p = ensure_server(port=port)
    test_is(p, None)
finally:
    await stop_server_a(port)

`ensure_server` returns the standard notebook/Solveit serving bundle: a live server, the FastHTML app it serves, the app route decorator, and a preview helper wired to that server.

Existing servers have priority. If a server is passed explicitly, it must be live and is used. Otherwise, an already-running server on the requested port is reused. If no matching server exists, a new one is started, using the provided app if any, otherwise creating a default app.

The returned preview helper always targets the returned server's port.

For the synchronous server returned here, `stop_server` stops the server and clears its notebook namespace aliases; use `await stop_server_a(...)` for asynchronous or general cleanup.

## esm loader

In [ ]:
#| export
_mod_nm_re = re.compile(r'[A-Za-z_][A-Za-z0-9_-]*\Z')
def _check_mod_nm(nm):
    if not isinstance(nm, str) or not _mod_nm_re.fullmatch(nm): raise ValueError(f"Invalid ES module name: {nm!r}")
    return nm

__mods__ = {}
def reg_mod(
    nm:str, # Kernel-global registry key
    src:str, # JavaScript module source served for this key
    v:int=0, # Version suffix; when > 0, the module is served under `<nm>?v=<v>`
) -> None:
    "Register or replace JavaScript source; entries persist in this kernel across app and server lifetimes."
    __mods__[_check_mod_nm(nm)+(f"_{v}" if v>0 else '')] = src
def _get_mod(
    nm:str, # Base module name from the request route
    v:int=0, # Requested version
    debugger:bool=False, # Prefix existing module source with `debugger;`
):
    "Return registered JavaScript as an HTTP response, or a diagnostic module when missing."
    src = __mods__.get(f'{nm}_{v}', __mods__.get(nm))
    if src is None: src,debugger = f'console.log("[{nm}] ES module version `{v}` not found");',False
    return Response(('debugger;\n' if debugger else '')+src, media_type='text/javascript')

In [ ]:
#| export
_mod_nm_re = re.compile(r'[A-Za-z_][A-Za-z0-9_-]*\Z')
def _check_mod_nm(nm):
    if not isinstance(nm, str) or not _mod_nm_re.fullmatch(nm): raise ValueError(f"Invalid ES module name: {nm!r}")
    return nm

__mods__ = {}
def reg_mod(
    nm:str, # Kernel-global registry key
    src:str, # JavaScript module source served for this key
    v:int=0, # Version suffix; when > 0, the module is served under `<nm>?v=<v>`
) -> None:
    "Register or replace JavaScript source; entries persist in this kernel across app and server lifetimes."
    __mods__[_check_mod_nm(nm)+(f"_{v}" if v>0 else '')] = src
def _get_mod(
    nm:str, # Base module name from the request route
    v:int=0, # Requested version
    debugger:bool=False, # Prefix existing module source with `debugger;`
):
    "Return registered JavaScript as an HTTP response, or a diagnostic module when missing."
    src = __mods__.get(f'{nm}_{v}', __mods__.get(nm))
    if src is None: src,debugger = f'console.log("[{nm}] ES module version `{v}` not found");',False
    return Response(('debugger;\n' if debugger else '')+src, media_type='text/javascript')

In [ ]:
for nm in ('', '1editor', 'a b', 'a/b', 'a?b', '../editor'):
    with expect_fail(ValueError, 'Invalid ES module name'): reg_mod(nm, '')

In [ ]:
#| exporti
def _bool_q(o): return str(o).lower() in ('1','true','yes','on')

async def esm_endpoint(req):
    pp = req.path_params
    try: nm = _check_mod_nm(pp['nm'])
    except ValueError as e: return Response(str(e), status_code=400)
    res = _get_mod(nm, int(req.query_params.get('v', 0)), _bool_q(req.query_params.get('debugger', False)))
    res.headers['Access-Control-Allow-Origin'] = '*'
    return res

In [ ]:
#| exporti
esm_routes = (Route('/esm/{nm}', esm_endpoint), )

In [ ]:
def setup_loader(app=None, mods=None):
    for nm,mod in (mods or {}).items(): reg_mod(nm,mod)

def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs, routes=esm_routes)
    else: app_kw = {**app_kw, 'routes': tuple(app_kw.get('routes', ()))+esm_routes}
    return FastHTML(**app_kw)

In [ ]:
# a = mk_app({})
# test_eq(any(r.path == '/esm/{nm}' for r in a.routes), True)
# # test_eq(any(r.path == '/esm/{nm}?v={v}' for r in a.routes), True)

In [ ]:
__mods__.clear()
a = mk_app({})
setup_loader(mods=dict(a='console.log("a")'))
cli = TestClient(a)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=1').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=2').text, 'console.log("a")')
reg_mod('a', 'console.log("a", 2)', 2)
test_eq(cli.get('/esm/a?v=2').text, 'console.log("a", 2)')

In [ ]:
__mods__.clear()
a = mk_app({})
cli = TestClient(a)

test_eq(cli.get('/esm/nope').headers['content-type'], 'text/javascript; charset=utf-8')
test_eq(cli.get('/esm/nope').text, 'console.log("[nope] ES module version `0` not found");')
test_eq(cli.get('/esm/nope?v=3&debugger=true').text, 'console.log("[nope] ES module version `3` not found");')

In [ ]:
#| exporti
def add_esm_routes(app):
    paths = {r.path for r in app.routes}
    for r in esm_routes:
        if r.path not in paths: app.routes.append(r)
    return app

In [ ]:
#| export
def setup_loader(
    app:FastHTML|None=None, # App receiving the ESM routes once; omit to register source without modifying an app
    mods:dict[str,str]|None=None, # Module names and sources added to the kernel-global registry
) -> None:
    "Idempotently install `/esm/{nm}` on `app` and register or replace kernel-global module sources."
    if app is not None: add_esm_routes(app)
    for nm,mod in (mods or {}).items(): reg_mod(nm,mod)

In [ ]:
#| exporti
def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return add_esm_routes(FastHTML(**(app_kw or {})))

In [ ]:
__mods__.clear()
a = FastHTML()
setup_loader(a, dict(a='console.log("a")'))
cli = TestClient(a)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(sum(r.path == '/esm/{nm}' for r in a.routes), 1)
setup_loader(a)
test_eq(sum(r.path == '/esm/{nm}' for r in a.routes), 1)

In [ ]:
#| export
async def ensure_server_a(
    srv:JupyUvi|None=None, # Explicit live server; its app and port win over all creation options
    port:int|None=None, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:FastHTML|None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str,str]|None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict|None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    public=False, # set True to enforce a public port if `srv` is not given
    **jupyuvi_kw, # Additional `JupyUviAsync` options used only when creating a server; ignored when reusing one
) -> tuple[JupyUvi,FastHTML,Callable|None,Callable|None]:
    "Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."
    srv,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, public=public, host=host, **jupyuvi_kw)
    setup_loader(srv.app, mods)
    return (*srv_bundle(srv),)

def ensure_server(
    srv:JupyUvi|None=None, # Explicit live server; its app and port win over all creation options
    port:int|None=None, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:FastHTML|None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str,str]|None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict|None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    public=False, # set True to enforce a public port if `srv` is not given
    **jupyuvi_kw, # Additional `JupyUvi` options used only when creating a server; ignored when reusing one
) -> tuple[JupyUvi,FastHTML,Callable|None,Callable|None]:
    "Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."
    srv,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, public=public, host=host, **jupyuvi_kw)
    setup_loader(srv.app, mods)
    return (*srv_bundle(srv),)

In [ ]:
__mods__.clear()
try:
    srv_a,app_a,rt_a,_ = ensure_server(mods=dict(a='console.log("a")'))
    srv_b,appB,rt_b,_ = ensure_server(mods=dict(b='console.log("b")'))
    cli = TestClient(app_a)

    test_eq(cli.get('/esm/a').text, 'console.log("a")')
    test_eq(cli.get('/esm/a?v=9&debugger=true').text, 'debugger;\nconsole.log("a")')
    test_eq(cli.get('/esm/b').text, 'console.log("b")')
finally:
    await stop_server_a(srv_a.port)

In [ ]:
await stop_server_a()

port = free_port(public=True)
srv = JupyUviAsync(FastHTML(), port=port, host='127.0.0.1')
await srv.start()

try:
    with expect_fail(RuntimeError, 'not publicly bound'):
        await ensure_server_a(port=port, public=True)
finally:
    await stop_server_a(port)

### Local dialog server

For the usual local dialog/notebook case, use default settings for non-public port selection:

```python
srv,app,rt,p = await ensure_server_a()

@rt
def hello(): return 'hello'

cli = TestClient(app)
test_eq(cli.get('/hello').text, 'hello')
```

Here `p` is `None`: private ports cannot be previewed in a Solveit iframe or used to setup ES modules loader. This is the normal mode for local notebook/dialog work and direct ASGI tests, rather than deploying a web application.

### Public server

```python
srv,app,rt,p = await ensure_server_a(public=True)

@rt
def hello(): return 'hello'
```

`await ensure_server_a(public=True)` argument is equivalent to `await ensure_server_a(port=free_port(public=True), host='0.0.0.0')`.

Then:

```python
p('/hello')
```

or:

```python
P(hx_get='/hello', hx_trigger='load once')
```

Stop server:

```python
await stop_server_a(srv.port)
```

`ensure_server_a()` returns the server bundle `(srv, app, rt, p)`:

- `srv` controls the uvicorn lifecycle.
- `app` is the application actually being served.
- `rt` is its FastHTML route decorator.
- `p` previews routes through the returned server.

Selection precedence is:

1. An explicit live `srv`.
2. A live `JupyUvi` already bound to the requested port.
3. A newly created server.

When an explicit or discovered server is reused, `app`, `app_kw`, and `jupyuvi_kw` are ignored. For an explicit `srv`, its own port is authoritative and the requested `port` is ignored. `mods` are still registered, and the ESM route is installed on the reused app. `port=None` selects a free port outside Solveit’s public-port allocation.

In notebooks, prefer `await stop_server_a(srv.port)`, which supports synchronous and asynchronous servers and clears all namespace aliases to the stopped server.

### CRAFT-local server
> `ensure_server_a` is designed as reusable dialog infrastructure, rather than as a web-app deployment or a special kind of server lifetime.

A CRAFT can initialize a local server once for the dialog:

```python
srv,app,rt,_ = await ensure_server_a()
```

This creates or reuses a non-public FastHTML server and returns the `(srv, app, rt, None)` bundle. Later cells can add routes through `rt` or test `app` directly without server setup.

That covers the main ESM loader behavior:

- versioned and unversioned routes
- caller-provided app route injection
- no duplicate route injection
- `ensure_server`/`ensure_server_a` passing `mods`
- reused server receiving additional `mods`
- missing module fallback
- `debugger=true` only applying when module source exists

## test rig

In [ ]:
#| exporti
def log(msg, *args, **kwargs): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

In [ ]:
log = await logger()

In [ ]:
#| exporti
class LoggerHandler(logging.Handler):
    def __init__(self, log): self.log = log; super().__init__()
    def emit(self, record):
        try: self.log(f"{(record.levelname+':'):<10}{self.format(record)}")
        except Exception: self.handleError(record)

In [ ]:
got = []
h = LoggerHandler(got.append)
h.emit(logging.LogRecord('x', logging.WARNING, '', 1, 'careful', (), None))
test_eq(got, ['WARNING:  careful'])

In [ ]:
class BadLog:
    def __call__(self, x): raise RuntimeError('boom')

h = LoggerHandler(BadLog())
h.handleError = lambda record: setattr(h, 'handled', True)
h.emit(logging.LogRecord('x', logging.INFO, '', 1, 'msg', (), None))
test_eq(h.handled, True)

In [ ]:
#| export
def setup_uvi_logging(
    _log:Callable[[str],object]|None=None, # Callable receiving formatted log lines; defaults to this module's `log`
    level:int=logging.INFO, # Level assigned to the uvicorn, uvicorn.error, and uvicorn.access loggers
) -> Callable[[],None]:
    "Route uvicorn logs to `_log`; return caller-owned cleanup and use `log_config=None` to preserve the setup."
    _log = log if _log is None else _log
    _lh = LoggerHandler(_log)
    names = ('uvicorn', 'uvicorn.error', 'uvicorn.access')
    prev = {}
    for name in names:
        lg = logging.getLogger(name)
        prev[name] = lg.handlers[:],lg.propagate,lg.level,lg.disabled
        lg.handlers,lg.propagate,lg.disabled = [_lh],False,False
        lg.setLevel(level)
    def cleanup():
        for name,(handlers,propagate,level,disabled) in prev.items():
            lg = logging.getLogger(name)
            lg.handlers,lg.propagate,lg.disabled = handlers,propagate,disabled
            lg.setLevel(level)
    return cleanup

Verify all three uvicorn loggers are changed and then restored.

In [ ]:
names = ('uvicorn','uvicorn.error','uvicorn.access')
logs = [logging.getLogger(o) for o in names]
old = [(o.handlers[:],o.propagate,o.level,o.disabled) for o in logs]
cleanup = setup_uvi_logging(lambda x: None, level=logging.WARNING)

test_eq([o.level for o in logs], [logging.WARNING]*3)
test_eq([o.propagate for o in logs], [False]*3)
test_eq([o.disabled for o in logs], [False]*3)
test_eq([len(o.handlers) for o in logs], [1]*3)

cleanup()
test_eq([(o.handlers,o.propagate,o.level,o.disabled) for o in logs], old)

###### start-section

In [ ]:
@asynccontextmanager
async def serve_app(
    app:FastHTML|None=None, # FastHTML app to serve; if omitted, create one with `app_kw`
    port:int|None=None, # Port to serve on; `None` chooses a free non-public port
    srv:JupyUvi|None=None, # Explicit live server; reused and never owned by this context
    reuse:bool=True, # Reuse a live server discovered on `port`; otherwise raise if one exists
    after_start:Callable[[JupyUvi],object]|None=None, # Sync or async callback after setup; failure triggers owned-server cleanup
    before_stop:Callable[[JupyUvi],object]|None=None, # Sync or async callback during cleanup, before owned-server shutdown
    restore_routes:bool=True, # Restore the initial route list only for a server created by this context
    timeout:float=5, # Seconds to wait for an owned server's port to become free
    mods:dict[str,str]|None=None, # Sources registered in the kernel-global ESM registry
    app_kw:dict|None=None, # Keywords used only when creating the default FastHTML app
    public=False, # Set True to enforce a public port if `srv` is not given
    **jupyuvi_kw, # Additional `JupyUviAsync` options used only when creating a server
) -> AsyncIterator[JupyUvi]:
    "Async context manager yielding a live server; stop and optionally restore routes only when this context created it."
    srv,own = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=reuse, public=public, **jupyuvi_kw)
    routes = list(srv.app.routes) if own and restore_routes and hasattr(srv.app, 'routes') else None
    primary = None
    try:
        setup_loader(srv.app, mods)
        if after_start: await maybe_await(after_start(srv))
        yield srv
    except BaseException as e:
        primary = e
        raise
    finally:
        errs = []
        try:
            if own and before_stop: await maybe_await(before_stop(srv))
        except BaseException as e: errs.append(e)
        try:
            if routes is not None: srv.app.routes[:] = routes
        except BaseException as e: errs.append(e)
        try:
            if own:
                srv.server.should_exit = True
                deadline = (loop := asyncio.get_running_loop()).time() + timeout
                while not is_port_free(srv.port, srv.host):
                    if loop.time() > deadline: raise TimeoutError(f"Port {srv.host}:{srv.port} still in use after {timeout}s")
                    await asyncio.sleep(0.1)
        except BaseException as e: errs.append(e)
        if errs:
            if primary is None:
                if len(errs) == 1: raise errs[0]
                raise BaseExceptionGroup("serve_app cleanup failures", errs)
            for e in errs: primary.add_note(f"serve_app cleanup failed: {type(e).__name__}: {e}")

In [ ]:
#| export
@asynccontextmanager
async def serve_app(
    app:FastHTML|None=None, # FastHTML app to serve; if omitted, create one with `app_kw`
    port:int|None=None, # Port to serve on; `None` chooses a free non-public port
    srv:JupyUvi|None=None, # Explicit live server; reused and never owned by this context
    reuse:bool=True, # Reuse a live server discovered on `port`; otherwise raise if one exists
    after_start:Callable[[JupyUvi],object]|None=None, # Sync or async callback after setup; failure triggers owned-server cleanup
    before_stop:Callable[[JupyUvi],object]|None=None, # Sync or async callback during cleanup, before owned-server shutdown
    restore_routes:bool=True, # Restore the initial route list only for a server created by this context
    timeout:float=5, # Seconds to wait for an owned server's port to become free
    mods:dict[str,str]|None=None, # Sources registered in the kernel-global ESM registry
    app_kw:dict|None=None, # Keywords used only when creating the default FastHTML app
    public=False, # Set True to enforce a public port if `srv` is not given
    **jupyuvi_kw, # Additional `JupyUviAsync` options used only when creating a server
) -> AsyncIterator[JupyUvi]:
    "Yield a live server in a temporary notebook namespace scope; stop and optionally restore routes only when this context created it."
    from unittest.mock import patch as mock_patch
    ns = get_ipython().user_ns
    with mock_patch.dict(ns, {}):
        srv,own = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=reuse, public=public, **jupyuvi_kw)
        routes = list(srv.app.routes) if own and restore_routes and hasattr(srv.app, 'routes') else None
        primary = None
        try:
            setup_loader(srv.app, mods)
            if after_start: await maybe_await(after_start(srv))
            yield srv
        except BaseException as e:
            primary = e
            raise
        finally:
            errs = []
            try:
                if own and before_stop: await maybe_await(before_stop(srv))
            except BaseException as e: errs.append(e)
            try:
                if routes is not None: srv.app.routes[:] = routes
            except BaseException as e: errs.append(e)
            try:
                if own:
                    srv.server.should_exit = True
                    deadline = (loop := asyncio.get_running_loop()).time() + timeout
                    while not is_port_free(srv.port, srv.host):
                        if loop.time() > deadline: raise TimeoutError(f"Port {srv.host}:{srv.port} still in use after {timeout}s")
                        await asyncio.sleep(0.1)
            except BaseException as e: errs.append(e)
            if errs:
                if primary is None:
                    if len(errs) == 1: raise errs[0]
                    raise BaseExceptionGroup("serve_app cleanup failures", errs)
                for e in errs: primary.add_note(f"serve_app cleanup failed: {type(e).__name__}: {e}")

These test both restoration cases: an existing `srv` binding survives a scoped server, while a new `test_srv` alias and names created in the block are removed.

In [ ]:
test_srv_port = []
old_srv = srv
globals().pop('test_srv', None)
globals().pop('scoped_only', None)

async with serve_app() as srv:
    test_is(srv is old_srv, False)
    test_srv_port.append(srv.port)
    scoped_only = 'temporary'

test_is(srv, old_srv)
test_eq('test_srv' in globals(), False)
test_eq('scoped_only' in globals(), False)
test_eq(is_port_free(test_srv_port[0]), True)

In [ ]:
ports = []
async with serve_app() as srv:
    ports.append(srv.port)
    test_eq(await is_serving_a(srv), True)
    test_eq(srv.port >= 6551, True)
    test_eq(srv.port in pub_ports(), False)
test_eq(is_port_free(ports[0]), True)

Temporary live server and cleans it up.

In [ ]:
base_srv,base_app,_,_ = await ensure_server_a()
called = []

async with serve_app(port=base_srv.port, before_stop=called.append) as srv: test_is(srv, base_srv)

test_eq(called, [])
test_eq(await is_serving_a(base_srv), True)
await stop_server_a(base_srv.port)

Existing servers are reused and not owned.

In [ ]:
async with serve_app() as srv:
    n = len(srv.app.routes)
    async with serve_app(port=srv.port) as same:
        test_is(same, srv)
        @same.app.get('/tmp-route')
        def tmp_route(): return 'ok'
    test_eq(len(srv.app.routes), n+1)

Owned app routes are restored.

In [ ]:
class BadMods:
    def items(self): raise RuntimeError('setup boom')

err = None
try:
    async with serve_app(mods=BadMods()): pass
except RuntimeError as e: err = e

test_eq(str(err), 'setup boom')
# test_eq(is_port_free(srv.port), True)

srvs,ns,err = [],[],None
async def bad_before(srv): raise RuntimeError('before boom')

try:
    async with serve_app(before_stop=bad_before) as srv:
        srvs.append(srv)
        ns.append(len(srv.app.routes))
        srv.app.get('/tmp-route')(lambda: 'ok')
        raise ValueError('body boom')
except ValueError as e: err = e

test_eq(str(err), 'body boom')
test_eq(err.__notes__, ['serve_app cleanup failed: RuntimeError: before boom'])
test_eq(len(srvs[0].app.routes), ns[0])
test_eq(is_port_free(srvs[0].port), True)

started = asyncio.Event()
async def wait_forever():
    async with serve_app():
        started.set()
        await asyncio.Future()

task = asyncio.create_task(wait_forever())
await started.wait()
task.cancel()
err = None
try: await task
except asyncio.CancelledError as e: err = e

test_eq(type(err), asyncio.CancelledError)
# test_eq(is_port_free(srv.port), True)

Cleanup and route restoration still happen when teardown callback fails.

In [ ]:
# Add log_config=None to prevent Uvicorn calling logging.config.dictConfig() on startup
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

The `log_config=None` kwarg passed in the test message prevents uvicorn from overwriting the custom `LoggerHandler` logging setup with its own `dictConfig` on startup.

In [ ]:
setup_uvi_logging()
await log.show(True)

In [ ]:
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

In [ ]:
__mods__.clear()
srvs=[]
async with serve_app(mods=dict(t='console.log("t")')) as srv:
    srvs.append(srv)
    cli = TestClient(srv.app)
    test_eq(cli.get('/esm/t').text, 'console.log("t")')
    test_eq(sum(r.path == '/esm/{nm}' for r in srv.app.routes), 1)
test_eq(is_port_free(srvs[0].port), True)

In [ ]:
srvs=[]
async with serve_app(public=True) as srv:
    srvs.append(srv)
    test_eq(srv.port in pub_ports(), True)
    test_eq(srv.host, '0.0.0.0')
    test_eq(await is_serving_a(srv), True)

test_eq(is_port_free(srvs[0].port, '0.0.0.0'), True)

I see an inconsistency. `serve_app` reuse a server if `reuse=True`. Reusing is based on `port`. If a server is found, the contexct manager creates an alias: if the existing server is named `srv` and the context manager names it `test_srv`, the end result is two aliases to the same object. That's not a problem. However, if `reuse=False`,  `port=None` / `port=<int>`, a new server is created, and the context is named like an existing server, the existing server is overwritten.

To avoid that problem, I think our `serve_app` should behave like unittest context managers and not contaminate the namespace in any way.

**NOTE**: You already have all the needed context in place, this dialog **is** your context. Please don't waste tokens unnecessarily; they're very expensive.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`serve_app` cannot prevent an `as srv` target from overwriting an existing `srv`: that assignment is performed by Python, after `__aenter__` returns. Yielding a different object would not change that.

A namespace-clean `serve_app` therefore needs to **stop yielding the server**. Its existing `after_start` callback can receive the live server for route setup and assertions, while the body is used only for the scoped lifetime:

```python
async with serve_app(after_start=lambda srv: srv.app.get('/hello')(lambda: 'hello')):
    ...
```

That is a deliberate API change: tests and callers that currently use `async with serve_app() as srv:` would instead capture only what they explicitly need in `after_start` (for example, `app` or `port`), or use `ensure_server_a` when they need persistent, interactive server state.

The intended contract would be:

- `ensure_server_a`: returns and leaves persistent server state under caller-managed names.
- `serve_app`: owns a temporary server lifetime and does not return it into the notebook namespace.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 48897, "completion_tokens": 1265, "total_tokens": 50162, "cached_tokens": 48384, "reasoning_tokens": 1034, "cost": 0.0082833}
```


Well, I don't know about that. But I know for sure that unittest.mock several context manager don't overwrite the namespace. We're been hitting by that several times, for instance, doing `test_eq()` outside the context manager scope with variables defined inside the block.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You are right. A context manager can preserve the notebook namespace by snapshotting it in `__aenter__`—before Python assigns the `as srv` target—and restoring it in `__aexit__`.

That is the relevant pattern here. `serve_app` should treat the body as a temporary scope:

- an existing `srv` binding is restored after the block;
- a new `test_srv` binding introduced by `as test_srv` is removed after the block;
- temporary names created inside the block should also be removed if the aim is strict namespace cleanliness.

The mechanism is a snapshot of `IPython.get_ipython().user_ns`, followed by restoring the pre-entry mapping in `finally`. It needs to exempt IPython’s own persistent bookkeeping names, but it can otherwise restore every overwritten binding and delete names absent from the snapshot.

So the existing API can remain:

```python
async with serve_app() as srv:
    ...
```

while ensuring that `srv` exists only for the context’s lifetime in the notebook namespace. This matches the scoped-resource expectation rather than requiring callers to avoid the `as` form.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 49234, "completion_tokens": 751, "total_tokens": 49985, "cached_tokens": 48384, "reasoning_tokens": 516, "cost": 0.00794835}
```


Please, ensure `unittest` behavior and the mechanism used is indeed that. Give me references.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

- ⏳ `web_search(type="search", queries="['site:docs.python.org/3/library/unittes…", query="site:docs.python.org/3/library/unittest.…")` ⏳


```json {.tool}
{
  "id": "ws_0f049a10b6885504016a93f4f7288c87d184d220ac97d1f108",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:docs.python.org/3/library/unittest.mock.html patch context manager \"with patch\"",
      "site:github.com/python/cpython Lib/unittest/mock.py _patch __enter__ __exit__"
    ],
    "query": "site:docs.python.org/3/library/unittest.mock.html patch context manager \"with patch\""
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="search", queries="['site:docs.python.org/3/reference/compo…", query="site:docs.python.org/3/reference/compoun…")` ⏳


```json {.tool}
{
  "id": "ws_0f049a10b6885504016a93f4fdf63c87d18770fd5e94d5f2b8",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:docs.python.org/3/reference/compound_stmts.html with statement \"TARGET\" \"__enter__\"",
      "site:docs.python.org/3/reference/simple_stmts.html assignment statement namespace binding"
    ],
    "query": "site:docs.python.org/3/reference/compound_stmts.html with statement \"TARGET\" \"__enter__\""
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="find_in_page", pattern="def __enter__(self):", url="https://github.com/python/cpython/blob/m…")` ⏳


```json {.tool}
{
  "id": "ws_0f049a10b6885504016a93f50613c487d181cb6b409aade088",
  "name": "web_search",
  "args": {
    "type": "find_in_page",
    "pattern": "def __enter__(self):",
    "url": "https://github.com/python/cpython/blob/main/Lib/unittest/mock.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="open_page", url="https://raw.githubusercontent.com/python…")` ⏳


```json {.tool}
{
  "id": "ws_0f049a10b6885504016a93f50929f487d1bcd2b4bdb63034ab",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="find_in_page", pattern="def __enter__(self):", url="https://raw.githubusercontent.com/python…")` ⏳


```json {.tool}
{
  "id": "ws_0f049a10b6885504016a93f50b82e487d1af0c96fdddf3d487",
  "name": "web_search",
  "args": {
    "type": "find_in_page",
    "pattern": "def __enter__(self):",
    "url": "https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```





My prior claim was too broad. The relevant `unittest.mock` mechanism is specifically **`patch.dict`**, not ordinary context-manager behavior or `patch()`.

`patch.dict(mapping, ...)` copies the entire target mapping on entry, applies its temporary changes, then clears and restores that original mapping on exit. CPython implements this as:

1. `original = in_dict.copy()`
2. apply temporary values
3. on exit: clear the mapping, then update it from `original` ([raw.githubusercontent.com](https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py))

So,[*](https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py "") if the mapping is a notebook’s `user_ns`, it restores prior bindings and removes bindings introduced during the block—including an `as srv` target, because Python assigns that target *after* `__aenter__` returns and before the suite runs. The language reference specifies that exact ordering for both `with` and `async with`. ([docs.python.org](https://docs.python.org/3/reference/compound_stmts.html?utm_source=openai))

By contrast, ordinary[*](https://docs.python.org/3/reference/compound_stmts.html?utm_source=openai "8. Compound statements — Python 3.14.7 documentation") `unittest.mock.patch()` only restores the **specific attribute it patches**; it does not snapshot or restore arbitrary local/global names. ([docs.python.org](https://docs.python.org/3.16/library/unittest.mock.html?utm_source=openai))

Therefore, the[*](https://docs.python.org/3.16/library/unittest.mock.html?utm_source=openai "unittest.mock — mock object library — Python 3.16.0a0 documentation") design you described is sound if the intended contract is a genuinely temporary notebook scope:

```python
async with serve_app() as srv:
    ...
```

Implementation-wise, `serve_app` should snapshot the IPython user namespace immediately on entry and restore it in `finally`, using the same clear-and-repopulate pattern as `patch.dict`. That restores an existing `srv` after the block and removes a newly introduced `test_srv`.

The trade-off is intentional: **every** ordinary name created or rebound inside the `async with` suite is discarded or restored on exit. That matches `patch.dict(user_ns)` semantics and explains why assertions outside such a scope cannot rely on names assigned within it.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 84480, "completion_tokens": 1663, "total_tokens": 86143, "cached_tokens": 3328, "reasoning_tokens": 1253, "cost": 0.013919250000000001}
```


including aliases, `test_srv` is removed.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

- ⏳ `web_search(type="open_page", url="https://raw.githubusercontent.com/python…")` ⏳


```json {.tool}
{
  "id": "ws_08d5af1aa3f5146d016a93f67e597c87d2a50d09fe0ebeb5c8",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="find_in_page", pattern="class _patch_dict", url="https://raw.githubusercontent.com/python…")` ⏳


```json {.tool}
{
  "id": "ws_08d5af1aa3f5146d016a93f6825e8887d295f808590cb4e902",
  "name": "web_search",
  "args": {
    "type": "find_in_page",
    "pattern": "class _patch_dict",
    "url": "https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```

Correct. The important case is:

```python
async with serve_app() as test_srv:
    ...
```

If `test_srv` did **not** exist on entry, restoring the pre-entry namespace removes it entirely on exit. If `srv` already referred to the same reused server, `srv` remains as it was; the temporary `test_srv` alias disappears.

That is exactly `patch.dict` behavior: it saves a copy of the whole mapping on entry, then on exit clears the mapping and restores that saved copy. Any bindings created during the block—including the `as test_srv` binding—are absent from the saved copy and therefore removed. ([raw.githubusercontent.com](https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py))

The ordering makes it viable[*](https://raw.githubusercontent.com/python/cpython/main/Lib/unittest/mock.py ""): for `async with`, `__aenter__` completes first, then Python assigns its returned value to `TARGET`; on exit, `__aexit__` can restore the snapshot after that assignment and the body have run. ([docs.python.org](https://docs.python.org/3/reference/compound_stmts.html))[*](https://docs.python.org/3/reference/compound_stmts.html "8. Compound statements — Python 3.14.7 documentation")

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 71280, "completion_tokens": 598, "total_tokens": 71878, "cached_tokens": 56576, "reasoning_tokens": 373, "cost": 0.011140500000000001}
```


Ok, let's see this version of `serve_app`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Replace `serve_app` with this version. It snapshots the IPython user namespace before yielding; on exit, `mock_patch.dict` restores that snapshot, removing temporary aliases such as `test_srv` and restoring any overwritten names.

```python
#| export
@asynccontextmanager
async def serve_app(
    app:FastHTML|None=None,
    port:int|None=None,
    srv:JupyUvi|None=None,
    reuse:bool=True,
    after_start:Callable[[JupyUvi],object]|None=None,
    before_stop:Callable[[JupyUvi],object]|None=None,
    restore_routes:bool=True,
    timeout:float=5,
    mods:dict[str,str]|None=None,
    app_kw:dict|None=None,
    public=False,
    **jupyuvi_kw,
) -> AsyncIterator[JupyUvi]:
    "Yield a live server in a temporary notebook namespace scope."
    from unittest.mock import patch as mock_patch
    ns = get_ipython().user_ns
    with mock_patch.dict(ns, {}):
        srv,own = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=reuse, public=public, **jupyuvi_kw)
        routes = list(srv.app.routes) if own and restore_routes and hasattr(srv.app, 'routes') else None
        primary = None
        try:
            setup_loader(srv.app, mods)
            if after_start: await maybe_await(after_start(srv))
            yield srv
        except BaseException as e:
            primary = e
            raise
        finally:
            errs = []
            try:
                if own and before_stop: await maybe_await(before_stop(srv))
            except BaseException as e: errs.append(e)
            try:
                if routes is not None: srv.app.routes[:] = routes
            except BaseException as e: errs.append(e)
            try:
                if own:
                    srv.server.should_exit = True
                    deadline = (loop := asyncio.get_running_loop()).time() + timeout
                    while not is_port_free(srv.port, srv.host):
                        if loop.time() > deadline: raise TimeoutError(f"Port {srv.host}:{srv.port} still in use after {timeout}s")
                        await asyncio.sleep(0.1)
            except BaseException as e: errs.append(e)
            if errs:
                if primary is None:
                    if len(errs) == 1: raise errs[0]
                    raise BaseExceptionGroup("serve_app cleanup failures", errs)
                for e in errs: primary.add_note(f"serve_app cleanup failed: {type(e).__name__}: {e}")
```

This preserves the useful form:

```python
async with serve_app() as test_srv:
    test_eq(await is_serving_a(test_srv), True)
```

After the block, `test_srv` is removed if it was new. If `srv` existed before the block and was temporarily rebound by `as srv`, its original binding is restored. Consequently, tests that need a value outside the context should place it in an object created before entry, rather than assigning a new top-level name inside the block.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 58404, "completion_tokens": 2040, "total_tokens": 60444, "cached_tokens": 57600, "reasoning_tokens": 1364, "cost": 0.0102906}
```


Replace `serve_app` with this version. It snapshots the IPython user namespace before yielding; on exit, `mock_patch.dict` restores that snapshot, removing temporary aliases such as `test_srv` and restoring any overwritten names.

In [ ]:
# #| export
# @asynccontextmanager
# async def serve_app(
#     app:FastHTML|None=None, # FastHTML app to serve; if omitted, create one with `app_kw`
#     port:int|None=None, # Port to serve on; `None` chooses a free non-public port
#     srv:JupyUvi|None=None, # Explicit live server; reused and never owned by this context
#     reuse:bool=True, # Reuse a live server discovered on `port`; otherwise raise if one exists
#     after_start:Callable[[JupyUvi],object]|None=None, # Sync or async callback after setup; failure triggers owned-server cleanup
#     before_stop:Callable[[JupyUvi],object]|None=None, # Sync or async callback during cleanup, before owned-server shutdown
#     restore_routes:bool=True, # Restore the initial route list only for a server created by this context
#     timeout:float=5, # Seconds to wait for an owned server's port to become free
#     mods:dict[str,str]|None=None, # Sources registered in the kernel-global ESM registry
#     app_kw:dict|None=None, # Keywords used only when creating the default FastHTML app
#     public=False, # Set True to enforce a public port if `srv` is not given
#     **jupyuvi_kw, # Additional `JupyUviAsync` options used only when creating a server
# ) -> AsyncIterator[JupyUvi]:
#     "Yield a live server in a temporary notebook namespace scope; stop and optionally restore routes only when this context created it."
#     from unittest.mock import patch as mock_patch
#     ns = get_ipython().user_ns
#     with mock_patch.dict(ns, {}):
#         srv,own = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=reuse, public=public, **jupyuvi_kw)
#         routes = list(srv.app.routes) if own and restore_routes and hasattr(srv.app, 'routes') else None
#         primary = None
#         try:
#             setup_loader(srv.app, mods)
#             if after_start: await maybe_await(after_start(srv))
#             yield srv
#         except BaseException as e:
#             primary = e
#             raise
#         finally:
#             errs = []
#             try:
#                 if own and before_stop: await maybe_await(before_stop(srv))
#             except BaseException as e: errs.append(e)
#             try:
#                 if routes is not None: srv.app.routes[:] = routes
#             except BaseException as e: errs.append(e)
#             try:
#                 if own:
#                     srv.server.should_exit = True
#                     deadline = (loop := asyncio.get_running_loop()).time() + timeout
#                     while not is_port_free(srv.port, srv.host):
#                         if loop.time() > deadline: raise TimeoutError(f"Port {srv.host}:{srv.port} still in use after {timeout}s")
#                         await asyncio.sleep(0.1)
#             except BaseException as e: errs.append(e)
#             if errs:
#                 if primary is None:
#                     if len(errs) == 1: raise errs[0]
#                     raise BaseExceptionGroup("serve_app cleanup failures", errs)
#                 for e in errs: primary.add_note(f"serve_app cleanup failed: {type(e).__name__}: {e}")

This preserves the useful form:

```python
async with serve_app() as test_srv:
    test_eq(await is_serving_a(test_srv), True)
```

After the block, `test_srv` is removed if it was new. If `srv` existed before the block and was temporarily rebound by `as srv`, its original binding is restored. Consequently, tests that need a value outside the context should place it in an object created before entry, rather than assigning a new top-level name inside the block.

Write some tests.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

These test both restoration cases: an existing `srv` binding survives a scoped server, while a new `test_srv` alias and names created in the block are removed.

```python
test_srv_port = []
old_srv = srv
globals().pop('test_srv', None)
globals().pop('scoped_only', None)

async with serve_app() as srv:
    test_is(srv is old_srv, False)
    test_srv_port.append(srv.port)
    scoped_only = 'temporary'

test_is(srv, old_srv)
test_eq('test_srv' in globals(), False)
test_eq('scoped_only' in globals(), False)
test_eq(is_port_free(test_srv_port[0]), True)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54393, "completion_tokens": 1704, "total_tokens": 56097, "cached_tokens": 3328, "reasoning_tokens": 1552, "cost": 0.009436950000000003}
```


These test both restoration cases: an existing `srv` binding survives a scoped server, while a new `test_srv` alias and names created in the block are removed.

In [ ]:
# test_srv_port = []
# old_srv = srv
# globals().pop('test_srv', None)
# globals().pop('scoped_only', None)
# 
# async with serve_app() as srv:
#     test_is(srv is old_srv, False)
#     test_srv_port.append(srv.port)
#     scoped_only = 'temporary'
# 
# test_is(srv, old_srv)
# test_eq('test_srv' in globals(), False)
# test_eq('scoped_only' in globals(), False)
# test_eq(is_port_free(test_srv_port[0]), True)

I think reusing a server is probably not worth the bother generally, very few use cases that could be served by one-off local test rigs. `app` reuse is more useful though, but I would create a separate context manager for the routes cleanup use case, because it's a useful functionality: quickly test some routes in a existing app and get rid of them afterwards with no route polution, restoring the state.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That separation is cleaner.

- `serve_app` should create and own a fresh server; no server reuse, no route restoration.
- App reuse should be explicit through a separate scoped route fixture.
- `ensure_server_a` remains the persistent dialog-infrastructure helper.

A route-scoping context manager could have this contract:

```python
async with scoped_routes(app) as rt:
    @rt('/temporary')
    def temporary(): return 'ok'
```

On exit, it restores `app.routes` to the exact pre-entry list. This supports direct `TestClient(app)` checks and testing against a live canonical app without permanently accumulating test routes.

It should deliberately restore only the route list. Headers, `app.state`, middleware, handler-closure state, and `__mods__` are distinct mutable state and should not be silently rolled back under a helper named for routes.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 60201, "completion_tokens": 316, "total_tokens": 60517, "cached_tokens": 58624, "reasoning_tokens": 134, "cost": 0.009267150000000002}
```


Ok, `serve_app` contract is then:

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`serve_app` contract:

- It is a **scoped server fixture**: it starts one server for the duration of the `async with` block and stops it on exit.
- It selects a **fresh port before startup**, then delegates setup to `ensure_server_a`.
  - `port=None, public=False`: choose a free private port.
  - `port=None, public=True`: choose a free public port.
  - Explicit `port`: it must be free or `serve_app` raises.
  - A port in `pub_ports()` implies public binding.
- It does **not intentionally reuse** a running server. The selected port is checked before calling `ensure_server_a`; in the single-kernel model, that prevents namespace-level reuse.
- It creates a fresh default FastHTML app when `app=None`. A supplied `app` is borrowed and served as-is; mutations to that app are not rolled back.
- `mods` are installed through the normal ESM setup path. Since `__mods__` is kernel-global, those registrations are not isolated or restored.
- `after_start` and `before_stop` may be sync or async and receive the newly started server.
- It yields the server so the block can inspect it or use `srv.app`.
- It uses the temporary namespace scope, so names introduced or rebound inside the block—including the `as test_srv` target—are removed or restored when the block exits.
- It does not restore routes, headers, middleware, application state, or module-registry entries. A separate route-scoping helper should own the route restoration contract.
- `ensure_server_a` remains the persistent dialog-infrastructure API; `serve_app` is for scoped tests and experiments.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 62016, "completion_tokens": 625, "total_tokens": 62641, "cached_tokens": 59648, "reasoning_tokens": 268, "cost": 0.00977115}
```


###### end-section

In [ ]:
#| export
@asynccontextmanager
async def serve_app(
    app:FastHTML|None=None, # App to serve; omit to create a default app
    port:int|None=None, # Exact port to bind; omit to select a free private/public port
    after_start:Callable[[JupyUvi],object]|None=None, # Sync or async callback after server startup
    before_stop:Callable[[JupyUvi],object]|None=None, # Sync or async callback before server shutdown
    mods:dict[str,str]|None=None, # JavaScript module sources registered for the app
    app_kw:dict|None=None, # Keywords used only when creating the default app
    public:bool=False, # Select a public port and public server binding
    **jupyuvi_kw, # Additional options passed when creating `JupyUviAsync`
) -> AsyncIterator[JupyUvi]:
    "Yield a newly created server in a temporary notebook namespace scope, then stop it."
    if port is None:
        host = '0.0.0.0' if public else '127.0.0.1'
        port = free_port(public=True, host=host) if public else free_port(6551, skip=pub_ports(), host=host)
    public = public or port in pub_ports()
    host = '0.0.0.0' if public else '127.0.0.1'
    if not is_port_free(port, host): raise RuntimeError(f"Port {port} is already in use")
    with mock_patch.dict(get_ipython().user_ns, {}):
        srv,app,_,_ = await ensure_server_a(
            app=app, port=port, mods=mods, app_kw=app_kw, public=public, **jupyuvi_kw
        )
        try:
            if after_start: await maybe_await(after_start(srv))
            yield srv
        finally:
            try:
                if before_stop: await maybe_await(before_stop(srv))
            finally: await srv.stop()

In [ ]:
ports = []
globals().pop('test_srv', None)

async with serve_app() as test_srv:
    ports.append(test_srv.port)
    test_eq(await is_serving_a(test_srv), True)

test_eq(is_port_free(ports[0]), True)
test_eq('test_srv' in globals(), False)

The first test establishes the fresh-server and temporary-alias contracts. Next pin down the central safety property: the canonical dialog `srv` remains unchanged.

In [ ]:
try:
    assert srv is None # right now, in this dialog `srv` is None.
    srv,_,_,_ = await ensure_server_a()

    async with serve_app() as test_srv:
        test_is(test_srv is srv, False)
        test_eq(await is_serving_a(srv), True)

    test_eq(await is_serving_a(srv), True)
    test_eq('test_srv' in globals(), False)
finally:
    await stop_server_a(srv.port)

The canonical-server test confirms `serve_app` creates a distinct server and leaves the existing dialog server live. Next test the no-reuse rule for an occupied explicit port:

In [ ]:
port = free_port(6551, skip=pub_ports())
errs = []

async with serve_app(port=port):
    try:
        async with serve_app(port=port): pass
    except RuntimeError as e: errs.append(str(e))

test_eq(errs, [f"Port {port} is already in use"])
test_eq(is_port_free(port), True)

The supplied-app contract is covered. Next verify callback order and cleanup:

In [ ]:
events,ports = [],[]

async def after(srv): events.append(('after', srv.port))
def before(srv): events.append(('before', srv.port))

async with serve_app(after_start=after, before_stop=before) as test_srv:
    ports.append(test_srv.port)
    test_eq(events, [('after', test_srv.port)])

test_eq(events, [('after', ports[0]), ('before', ports[0])])
test_eq(is_port_free(ports[0]), True)

The normal callback sequence is covered. Next verify that an `after_start` failure still runs `before_stop` and releases the port:

In [ ]:
events,ports,err = [],[],None

async def after(srv):
    ports.append(srv.port)
    events.append('after')
    raise RuntimeError('after boom')

def before(srv): events.append('before')

try:
    async with serve_app(after_start=after, before_stop=before): pass
except RuntimeError as e: err = e

test_eq(str(err), 'after boom')
test_eq(events, ['after', 'before'])
test_eq(is_port_free(ports[0]), True)

The callback-failure path is covered. Now verify a failure in the context body still stops the server and removes the temporary alias:

In [ ]:
ports,err = [],None
globals().pop('test_srv', None)

try:
    async with serve_app() as test_srv:
        ports.append(test_srv.port)
        raise ValueError('body boom')
except ValueError as e: err = e

test_eq(str(err), 'body boom')
test_eq(is_port_free(ports[0]), True)
test_eq('test_srv' in globals(), False)

The body-failure path is covered. Next verify `before_stop` failures do not prevent shutdown:

In [ ]:
ports,err = [],None

def before(srv):
    ports.append(srv.port)
    raise RuntimeError('before boom')

try:
    async with serve_app(before_stop=before): pass
except RuntimeError as e: err = e

test_eq(str(err), 'before boom')
test_eq(is_port_free(ports[0]), True)

This distinguishes app/server isolation from module-registry isolation.

In [ ]:
setup_uvi_logging()
await log.show(True)

In [ ]:
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

###### start-section

Done with the tests?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. The refactored `serve_app` now has coverage for:

- fresh private-server startup, shutdown, and temporary namespace alias removal;
- isolation from the canonical dialog server;
- occupied explicit ports raising rather than reusing;
- public-port selection and cleanup;
- supplied-app identity and caller ownership;
- sync/async lifecycle callback order;
- cleanup after `after_start`, body, and `before_stop` failures;
- ESM route/module installation;
- the intentional kernel-global persistence of `__mods__`.

I would stop here. Further tests would mostly repeat the same lifecycle and port-selection mechanics.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 65886, "completion_tokens": 202, "total_tokens": 66088, "cached_tokens": 64768, "reasoning_tokens": 73, "cost": 0.0100344}
```


`serve_app` current version is a thin wrapper over `ensure_serve_a`, adding only `after_start`/`before_stop`/`mods`. `app`, `port`, `app_kw`, `public`, `jupyuvi_kw` semantics are exactly the same as `ensure_server_a`, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Not exactly.

`serve_app` forwards `app`, `app_kw`, `public`, `mods`, and `**jupyuvi_kw` to `ensure_server_a`, so their **creation-path semantics** are the same.

Its `port` semantics deliberately differ:

- `ensure_server_a(port=x)` reuses a live `JupyUvi` already found on `x`.
- `serve_app(port=x)` first requires `x` to be free, so it raises rather than reusing an existing server.
- With `port=None`, both choose a free private port by default or a public port with `public=True`; `serve_app` merely selects that port before delegation.

`serve_app` also adds more than the two callbacks and `mods`:

- temporary `user_ns` scope via `mock_patch.dict`;
- unconditional shutdown of the server it starts;
- its no-reuse / fresh-server constraint.

So it is a thin wrapper operationally, but not semantically identical: it preserves `ensure_server_a`’s app-creation configuration while narrowing server selection and adding scoped lifetime behavior.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 66134, "completion_tokens": 484, "total_tokens": 66618, "cached_tokens": 64768, "reasoning_tokens": 252, "cost": 0.0102831}
```


Right. So we shouldn't test what's already tested in `ensure_server_a` and focus only on what's different. At least for correctness, it's true that test are also documentation.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly. `serve_app` correctness tests should cover only its added contract:

- explicit occupied port raises rather than reusing;
- a fresh scoped server is stopped on exit;
- temporary namespace bindings are restored;
- `after_start` / `before_stop` order and failure cleanup;
- canonical dialog server remains unaffected;
- any intentionally distinct `mods` behavior, if applicable.

Remove tests that merely repeat `ensure_server_a` coverage for default app creation, public-port selection, supplied-app forwarding, and ESM installation. Those may remain as a short documentation example if they teach `serve_app` usage, but should not be duplicated as correctness tests.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 66454, "completion_tokens": 171, "total_tokens": 66625, "cached_tokens": 64768, "reasoning_tokens": 34, "cost": 0.01009635}
```


Let's now focus on the `reuse_routes(app)` helper.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`reuse_routes` can be a synchronous, narrowly scoped route fixture:

- snapshot `app.routes` on entry;
- yield the app’s route decorator;
- restore the exact original route list on exit;
- leave headers, middleware, app state, handlers, and `__mods__` untouched.

```python
from contextlib import contextmanager
```

```python
#| export
@contextmanager
def reuse_routes(
    app:FastHTML, # App whose route list is temporarily extended
) -> Callable:
    "Yield `app.route`, restoring its initial routes when the context exits."
    routes = list(app.routes)
    try: yield app.route
    finally: app.routes[:] = routes
```

Its intended form is:

```python
async with serve_app() as test_srv:
    with reuse_routes(test_srv.app) as rt:
        @rt
        def temporary(): return 'ok'
```

This makes route cleanup explicit without claiming to roll back other application state.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 66820, "completion_tokens": 725, "total_tokens": 67545, "cached_tokens": 65792, "reasoning_tokens": 516, "cost": 0.01056675}
```


###### end-section

### Temporary server and testing

Use `serve_app` when a server should exist only for a block:

```python
async with serve_app() as srv:
    @srv.app.get('/hello')
    def hello(): return 'hello'

    cli = TestClient(srv.app)
    test_eq(cli.get('/hello').text, 'hello')
```

A server created by the context is owned by it: cleanup runs after normal exit, setup or body errors, and cancellation. By default, routes added to an owned app during the block are removed on exit.

A reused server is not owned by the context. It remains live afterward, and route changes made to it persist.

The context yields only the live server; use `srv.app`, `srv.port`, and other server attributes inside the block. In tests, `port=None` avoids persistent and publicly allocated ports.

`TestClient(srv.app)` tests the ASGI application directly and does not prove that uvicorn is reachable over the network. Use `await is_serving_a(srv)` when network availability is part of the contract.

### Temporary server and testing

Use `serve_app` when a server should exist only for a block:

```python
async with serve_app() as test_srv:
    @test_srv.app.get('/hello')
    def hello(): return 'hello'

    cli = TestClient(test_srv.app)
    test_eq(cli.get('/hello').text, 'hello')
```

`serve_app` always starts a fresh server and owns its lifetime. It stops that server after normal exit, setup or body errors, and cancellation. An explicitly requested port must be free; an occupied port raises rather than reusing its existing server.

The context temporarily scopes the notebook namespace. A new alias such as `test_srv` is removed on exit; an existing name temporarily rebound by the context is restored.

Route changes made to `test_srv.app` persist on a caller-supplied app.

The context yields the live server; use `test_srv.app`, `test_srv.port`, and its other attributes inside the block. `port=None` selects a free non-public port by default.

### Uvicorn logging

`setup_uvi_logging` routes the `uvicorn`, `uvicorn.error`, and `uvicorn.access` loggers to a callable and returns a cleanup function that restores their previous configuration.

Pass `log_config=None` when starting uvicorn; otherwise uvicorn may replace the custom logger configuration.

```python
cleanup = setup_uvi_logging(my_log)
try:
    async with serve_app(log_config=None) as srv:
        ...
finally:
    cleanup()
```

Logging configuration is process-global rather than server-owned, so the caller is responsible for cleanup.

## inject mod
Turn JavaScript stored in dialog messages into browser-loadable ES modules.

In [ ]:
#| export
async def get_js(
    id:str|None=None, # JavaScript message id; defaults to the preceding message
    dname:str='', # Optional dialog name containing the message
) -> str|None:
    "Strip the leading `%%js` or `%%javascript` line from a dialog message and return its remaining source."
    fl,_,src = (await read_msg(n=0 if id else -1, id=id, dname=dname)).content.partition('\n')
    if fl.lower() not in ('%%javascript', '%%js'): return None
    return src

In [ ]:
test_eq(await get_js(), 'a')

In [ ]:
test_is(await get_js(), None)

In [ ]:
#| exporti
def _py_assign(nm:str, src:str):
    "Return a Python assignment that round-trips `src`; reject invalid variable names."
    if not nm.isidentifier() or keyword.iskeyword(nm): raise ValueError(f"Invalid Python identifier: {nm!r}")
    return f'{nm} = {src!r}'

In [ ]:
src = r"""const single = \"'\"
const triple = "'''"
const path = String.raw`C:\tmp\file`
const tmpl = `line 1
line 2`
""" + "\\"
ns = {}
exec(_py_assign('_src', src), ns)
test_eq(ns['_src'], src)

for nm in ('not-valid', 'class', ''):
    with expect_fail(ValueError, 'Invalid Python identifier'): _py_assign(nm, src)

In [ ]:
#| export
@delegates(link_msg, but='content,msg_type')
async def add_js_msg(
    nm:str, # Valid Python identifier assigned to safely serialized source; invalid names raise `ValueError`
    id:str=None, # JavaScript source message; defaults to the preceding message and does not change the linkage anchor
    dname:str='', # Dialog to get info for; the linked code message is created in the current dialog
    **kwargs, # Delegated `link_msg` placement, execution, and display options; output is always a code message
) -> str:
    "Extract JavaScript from a dialog message and create or update a linked Python code message containing a safely serialized assignment."
    if (src := await get_js(id, dname)) is not None:
        return await link_msg(_py_assign(nm, src), msg_type='code', **kwargs)
    raise ValueError(f"Message #{id or (await read_msg()).id} does not contain Javascript source")

In [ ]:
await add_js_msg('_test_js', i_collapsed=True);

In [ ]:
_test_js = "const test = {'hello':'world'}\nglobalThis._test = test\nexport {test}"
#| linkedto: d954aa42

In [ ]:
v,dname = None,find_dname()
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
'&'.join(q)

In [ ]:
v=1
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
'&'.join(q)

In [ ]:
nm = 'a'
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
url

In [ ]:
nm = 'a'
q = []
url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
url

In [ ]:
@delegates(serve_app)
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int|None=None, # Numeric version selecting registered `<nm>_<v>` before falling back to `<nm>`,
    dname:str='', # Dialog owning the module; will be appended as query param if present
    **kwargs
) -> object:
    "Ensure the server, import `/esm/<nm>` in the active Solveit browser, then release any temporary server; browser and registry state persist."
    q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
    url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
    async with serve_app(**kwargs) as _: return await js_eval_a(f"return await import({json.dumps(url)})")

In [ ]:
# @asynccontextmanager
# async def fake_serve_app(*args, **kwargs):
#     events.append(('enter', args, kwargs))
#     try: yield None
#     finally: events.append(('exit',))
# 
# async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

In [ ]:
# events = []
# 
# with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a):
#     test_eq(await load_mod('demo', 2), 'ok')
# 
# test_eq(events, [('enter', (), {}), ('eval', 'return await import("/esm/demo?v=2")'), ('exit',)])

In [ ]:
# events = []
# 
# with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a):
#     test_eq(await load_mod('demo', dname='a b/c'), 'ok')
# 
# test_eq(events, [('enter', (), {}), ('eval', 'return await import("/esm/demo?dlg_name=a%20b%2Fc")'), ('exit',)])

We've already tested `serve_app`, right? If not, we should. We trust `js_eval_a` implicitly; if detect any issue with it, we should open a PR with Answer.AI. We trust python handing of strings and URLs, right? Then, what are we testing here?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`serve_app` has already been tested extensively above: fresh lifecycle, isolation from the dialog server, occupied-port failure, callbacks, cleanup after setup/body failures, and logging.

These two tests are not testing `serve_app`, `js_eval_a`, Python string handling, or URL encoding itself. They test `load_mod`’s small integration contract:

- It opens the temporary server context before evaluating the browser import, and exits it afterward.
- It constructs the expected import URL:
  - version: `"/esm/demo?v=2"`
  - dialog name: `"/esm/demo?dlg_name=a%20b%2Fc"`
- It passes that URL to `js_eval_a` as a JavaScript dynamic import.

The mocks are appropriate because the dependencies’ behavior is outside this function’s responsibility. The exact-string assertions pin down the part `load_mod` owns: query assembly and call ordering.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52574, "completion_tokens": 656, "total_tokens": 53230, "cached_tokens": 3328, "reasoning_tokens": 467, "cost": 0.008378100000000001}
```


But then we should have tested the behavior of lines 10-11 before assembling the function:
```python
v,dname = None,find_dname()
q = filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else ''))
'&'.join(q)

# 'dlg_name=prj%2Fpote%2Fnbs%2F02_server'
```

```python
v=1
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
'&'.join(q)

# 'v=1&dlg_name=prj%2Fpote%2Fnbs%2F02_server'
```

```python
nm = 'a'
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
url

# '/esm/a?v=1&dlg_name=prj%2Fpote%2Fnbs%2F02_server'
```

To assess that at long last we know the behavior of `filter`, string interpolation and addition, `quote`, and `join` in a previous mini-exploration section. 

And then assemble the function with the same exact argument names, confident that it'll work as expected, because I've just tested it.

This stuff is so basic that I wouldn't check it at all, but that just me. What I would do is **document** the contract visually with a couple of examples. This is the literate programming way that Solveit/Jupyter enable.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. Those cells establish the query-construction behavior before `load_mod` combines it with server setup and browser evaluation. They are useful as **documentation examples**, not correctness tests of Python primitives.

The appropriate literate structure is:

1. A short exploratory section that visibly demonstrates the URL forms:
   - no version, dialog name only;
   - version plus dialog name;
   - complete `/esm/{nm}` URL.
2. The `load_mod` implementation, using the same variable names and expressions.
3. Focused tests only for the function-owned integration contract: it enters/exits `serve_app` and sends that constructed URL to `js_eval_a`.

That leaves `filter`, f-strings, `quote`, and `join` as trusted language/library machinery while preserving clear, executable examples of the contract users need to understand.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 53561, "completion_tokens": 346, "total_tokens": 53907, "cached_tokens": 3328, "reasoning_tokens": 114, "cost": 0.008293650000000001}
```


Right. Let's see that integration contract, more as documentation/simple illustrative full examples than correctness, because we know that `serve_app` and `js_eval_a` work.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Replace the mock-based checks with real, visible examples. This shows the end-to-end contract: register source, load it in the browser, and observe its effect.

```python
reg_mod('demo', 'document.documentElement.dataset.demo = "loaded"; export const status = "loaded"')

await load_mod('demo')
await js_eval_a('return document.documentElement.dataset.demo')
```

```python
reg_mod('demo_2', 'document.documentElement.dataset.demo = "version-2"; export const status = "version-2"')

await load_mod('demo', v=2)
await js_eval_a('return document.documentElement.dataset.demo')
```

The first illustrates the default `/esm/demo` import; the second illustrates that `v=2` selects the registered `demo_2` source.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 53826, "completion_tokens": 1723, "total_tokens": 55549, "cached_tokens": 51456, "reasoning_tokens": 1552, "cost": 0.00936615}
```


In [ ]:
reg_mod('demo', 'document.documentElement.dataset.demo = "loaded"; export const status = "loaded"')

res = await load_mod('demo', public=True)
test_eq(hx.eval('document.documentElement.dataset.demo'), 'loaded')
res

In [ ]:
reg_mod('demo', 'document.documentElement.dataset.demo = "version-2"; export const status = "version-2"', v=2)

await load_mod('demo', v=2, public=True)
test_eq(hx.eval('document.documentElement.dataset.demo'), 'version-2')

In [ ]:
test_eq((await js_eval_a('''const mod = await import('/esm/demo?v=2')\nreturn mod''')).result, {'status': 'version-2'})

The module is cached; we don't need a running server.

As we need a public server to load the module the first time, `load_mod` ensure the loader is public. For ports other than 8000 we need a fully qualified URL.

In [ ]:
#| export
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int|None=None, # Numeric version selecting registered `<nm>_<v>` before falling back to `<nm>`
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> object:
    "Import `/esm/<nm>` in the active Solveit browser; return URL and module result. Need at least one public port free."
    _check_mod_nm(nm)
    async with serve_app(public=True) as srv:
        q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)  
        base = '' if srv.port == 8000 else f"https://{pub_domain(srv.port)}"
        url = f'{base}/esm/{nm}' + ('?' if q else '') + '&'.join(q)
        return url, await js_eval_a(f"return await import({json.dumps(url)})")

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The revised `load_mod` should next be checked for its public-server URL and encoded dialog name:

```python
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=8001)
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a,
                     pub_ports=lambda: (8000,8001), pub_domain=lambda port: 'p8001.solve.it.com'):
    test_eq(await load_mod('demo', 2, 'a b/c'), 'ok')

test_eq(events, [
    ('enter', (), dict(public=True)),
    ('eval', 'return await import("https://p8001.solve.it.com/esm/demo?v=2&dlg_name=a%20b%2Fc")'),
    ('exit',),
])
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52852, "completion_tokens": 738, "total_tokens": 53590, "cached_tokens": 3328, "reasoning_tokens": 516, "cost": 0.0084813}
```


The revised `load_mod` should next be checked for its public-server URL and encoded dialog name:

In [ ]:
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=8001)
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a,
                     pub_ports=lambda: (8000,8001), pub_domain=lambda port: 'p8001.solve.it.com'):
    test_eq(await load_mod('demo', 2, 'a b/c'), 'ok')

test_eq(events, [
    ('enter', (), dict(public=True)),
    ('eval', 'return await import("https://p8001.solve.it.com/esm/demo?v=2&dlg_name=a%20b%2Fc")'),
    ('exit',),
])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The non-default public-port URL and dialog-name encoding test passed. Next cover port `8000`, which should use the relative module URL:

```python
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=8000)
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a, pub_ports=lambda: (8000,)):
    test_eq(await load_mod('demo'), 'ok')

test_eq(events, [
    ('enter', (), dict(public=True)),
    ('eval', 'return await import("/esm/demo")'),
    ('exit',),
])
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 53368, "completion_tokens": 345, "total_tokens": 53713, "cached_tokens": 8448, "reasoning_tokens": 161, "cost": 0.008263949999999999}
```


The non-default public-port URL and dialog-name encoding test passed. Next cover port `8000`, which should use the relative module URL:

In [ ]:
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=8000)
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a, pub_ports=lambda: (8000,)):
    test_eq(await load_mod('demo'), 'ok')

test_eq(events, [
    ('enter', (), dict(public=True)),
    ('eval', 'return await import("/esm/demo")'),
    ('exit',),
])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The port-8000 case passed. Next verify that `load_mod` rejects a server that is not publicly exposed and does not attempt the browser import:

```python
events,err = [],None

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=6551)
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a, pub_ports=lambda: (8000,)):
    try: await load_mod('demo')
    except RuntimeError as e: err = e

test_eq(str(err), 'Port 6551 is not publicly exposed')
test_eq(events, [('enter', (), dict(public=True)), ('exit',)])
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 53806, "completion_tokens": 393, "total_tokens": 54199, "cached_tokens": 52480, "reasoning_tokens": 196, "cost": 0.008365649999999999}
```


The port-8000 case passed. Next verify that `load_mod` rejects a server that is not publicly exposed and does not attempt the browser import:

In [ ]:
@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=6551)
    finally: events.append(('exit',))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a, pub_ports=lambda: (8000,)):
    with expect_fail(RuntimeError, 'Port 6551 is not publicly exposed'):
        await load_mod('demo')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The non-public-server failure case passed. Next verify invalid module names fail before `serve_app` starts:

```python
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter',))
    yield AD(port=8000)

with mock_patch.dict(globals(), serve_app=fake_serve_app):
    with expect_fail(ValueError, 'Invalid ES module name'):
        await load_mod('../demo')

test_eq(events, [])
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54232, "completion_tokens": 198, "total_tokens": 54430, "cached_tokens": 8448, "reasoning_tokens": 92, "cost": 0.0082833}
```


The non-public-server failure case passed. Next verify invalid module names fail before `serve_app` starts:

In [ ]:
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter',))
    yield AD(port=8000)

with mock_patch.dict(globals(), serve_app=fake_serve_app):
    with expect_fail(ValueError, 'Invalid ES module name'):
        await load_mod('../demo')

test_eq(events, [])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The invalid-name test passed. Next verify a browser-import failure still exits the temporary server context:

```python
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=8000)
    finally: events.append(('exit',))

async def fake_js_eval_a(src):
    events.append(('eval', src))
    raise RuntimeError('import boom')

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a, pub_ports=lambda: (8000,)):
    with expect_fail(RuntimeError, 'import boom'):
        await load_mod('demo')

test_eq(events, [
    ('enter', (), dict(public=True)),
    ('eval', 'return await import("/esm/demo")'),
    ('exit',),
])
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54512, "completion_tokens": 445, "total_tokens": 54957, "cached_tokens": 53504, "reasoning_tokens": 259, "cost": 0.008510550000000002}
```


The invalid-name test passed. Next verify a browser-import failure still exits the temporary server context:

In [ ]:
events = []

@asynccontextmanager
async def fake_serve_app(*args, **kwargs):
    events.append(('enter', args, kwargs))
    try: yield AD(port=8000)
    finally: events.append(('exit',))

async def fake_js_eval_a(src):
    events.append(('eval', src))
    raise RuntimeError('import boom')

with mock_patch.dict(globals(), serve_app=fake_serve_app, js_eval_a=fake_js_eval_a, pub_ports=lambda: (8000,)):
    with expect_fail(RuntimeError, 'import boom'):
        await load_mod('demo')

test_eq(events, [
    ('enter', (), dict(public=True)),
    ('eval', 'return await import("/esm/demo")'),
    ('exit',),
])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The browser-import failure still closes the temporary server context. The remaining contract is an end-to-end import against a real public server and the active browser:

```python
reg_mod('test_load_mod', 'globalThis.__test_load_mod = 7; export const answer = 7')

mod = await load_mod('test_load_mod')

test_eq(mod['answer'], 7)
test_eq(await js_eval_a('return globalThis.__test_load_mod'), 7)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54953, "completion_tokens": 618, "total_tokens": 55571, "cached_tokens": 8448, "reasoning_tokens": 516, "cost": 0.008706450000000001}
```


The browser-import failure still closes the temporary server context. The remaining contract is an end-to-end import against a real public server and the active browser:

In [ ]:
reg_mod('test_load_mod', 'globalThis.__test_load_mod = 7; export const answer = 7')

url,mod = await load_mod('test_load_mod')

test_eq(mod['result']['answer'], 7)
test_eq((await js_eval_a('return globalThis.__test_load_mod'))['result'], 7)

That's all very interesting, very SaS best practices, but let's analyze what's `load_mode`really does in the context of this notebook:

1. `_check_mod_nm(nm)`
    We already trust this function, don't we?. No need to re-test here because `nm` is forwarded as is.
2. `async with serve_app(public=True) as srv:`
    `serve_app(public=True)` lifecycle is also trusted, nothing to test here. If we detect some problem pointing to `serve_app` we should test there not here.
3. `q = tuple(filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')))`
    this is vanilla python. Nothing to test here. If the caller sends invalid `v` / `dname` in the context of an url that's a problem of the caller, not this function.
4. `if srv.port not in pub_ports(): raise RuntimeError(f"Port {srv.port} is not publicly exposed")`
    `pub_ports` we confide in, right? We know `serve_app` raises correctly. So nothing to see here.
5. `base = '' if srv.port == 8000 else f"https://{pub_domain(srv.port)}"`
    We are confident python can interpolate string.
6. `url = f'{base}/esm/{nm}' + ('?' if q else '') + '&'.join(q)`
    We are confident python can interpolate string and add strings.
7. `return url, await js_eval_a(f"return await import({json.dumps(url)})")`
    We're confident `AnswerDotAI` can do this. But this is the only point that we should really test (more for documentation than correctness)

Am I missing something?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You are not missing anything material. For `load_mod`, the only non-trivial contract it adds is: **a registered module becomes importable in the active browser, and its exports are returned through `js_eval_a`.** The end-to-end test already documents and checks that.

The earlier tests mostly re-tested trusted dependencies or Python expression assembly. They increase maintenance cost without strengthening `load_mod`’s contract. If `serve_app`, `pub_ports`, URL construction, or module-name validation need coverage, their tests belong with those helpers.

Keep the real-browser test, ideally phrased as the public contract:

```python
reg_mod('test_load_mod', 'globalThis.__test_load_mod = 7; export const answer = 7')

mod = await load_mod('test_load_mod')

test_eq(mod['result']['answer'], 7)
test_eq((await js_eval_a('return globalThis.__test_load_mod'))['result'], 7)
```

That checks both exported-module access and execution of the imported browser code.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 55630, "completion_tokens": 349, "total_tokens": 55979, "cached_tokens": 8448, "reasoning_tokens": 132, "cost": 0.008606250000000001}
```


Right. Perhaps the only other end-to-end test could be an invalid module, mostly to see the error `js_eval_a` returns, if any.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That would document the browser-side failure path. Register a deliberately throwing module under a fresh name, then inspect the unmodified `js_eval_a` result:

```python
reg_mod('test_load_mod_error', 'throw new Error("test module failure")')

await load_mod('test_load_mod_error')
```

If `js_eval_a` returns a structured browser error, the cell output shows its shape. If it propagates the browser exception, the cell fails with that exception. Either outcome establishes the contract callers encounter.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 55932, "completion_tokens": 627, "total_tokens": 56559, "cached_tokens": 8448, "reasoning_tokens": 516, "cost": 0.008860050000000001}
```


In [ ]:
reg_mod('test_load_mod_error', 'throw new Error("test module failure")')

with expect_fail(DialogAPIError, 'Timed out'): await load_mod('test_load_mod_error')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The browser rejection never reaches `js_eval_a`’s result channel. The `await import(...)` rejects before `pushData(...)` executes, so `pop_data_a` waits 15 seconds and raises `DialogAPIError: Timed out`.

This establishes the current failure contract, but it is not a useful one: it discards the JavaScript error message (`test module failure`). The next design decision is whether `load_mod` should preserve browser import errors as structured returned data or re-raise them with the original message.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 56700, "completion_tokens": 394, "total_tokens": 57094, "cached_tokens": 54528, "reasoning_tokens": 280, "cost": 0.008800500000000001}
```


What'd be best in your opinion? I lean to fail often, fail fast, but I'm never sure.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Raise a Python exception from `load_mod`, preserving the browser error message. That fits “fail fast”: callers ask for a usable module; a failed import is not a normal result to inspect.

The timeout is the wrong failure mode because it is slow and hides the cause. `load_mod` should catch the browser-side rejection only long enough to marshal it back through `js_eval_a`, then immediately raise locally. Keep `js_eval_a` unchanged: this is a `load_mod`-specific requirement.

The intended behavior is:

```python
try: mod = await load_mod('test_load_mod_error')
except RuntimeError as e: print(e)
```

which should fail promptly with a message containing `test module failure`, rather than after 15 seconds with `DialogAPIError: Timed out`.

I would return only the error name/message by default. A browser stack trace is useful while debugging, but is noisy as the ordinary exception text.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 56891, "completion_tokens": 543, "total_tokens": 57434, "cached_tokens": 55552, "reasoning_tokens": 345, "cost": 0.008940900000000002}
```


That would be great, but unfortunately `js_eval_a` behavior is not in our hands.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

We do not need to change `js_eval_a`. `load_mod` can catch the rejection **inside the JavaScript expression** it passes to `js_eval_a`, turning it into ordinary serializable data before `js_eval_a` tries to send it back.

That preserves the existing success result shape and replaces the timeout with an immediate Python exception:

```python
res = await js_eval_a(f'''try {{
    return await import({json.dumps(url)})
}} catch (e) {{
    return {{__load_mod_error__: `${{e.name}}: ${{e.message}}`}}
}}''')

if err := res['result'].get('__load_mod_error__'): raise RuntimeError(err)
return res
```

For the throwing test module, this should raise promptly with:

```text
Error: test module failure
```

The only caveat is that `__load_mod_error__` is a reserved export name in practice. A more elaborate envelope avoids even that theoretical collision, but would change the successful return shape.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 57162, "completion_tokens": 730, "total_tokens": 57892, "cached_tokens": 56576, "reasoning_tokens": 516, "cost": 0.0091218}
```


Your're right. Let's see this version.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Replace `load_mod` with this version. It catches an import rejection in the browser, returns its name/message through the existing result channel, then raises immediately in Python.

```python
#| export
async def load_mod(
    nm:str,
    v:int|None=None,
    dname:str='',
) -> object:
    "Import a registered ES module in the active Solveit browser and return its exports."
    _check_mod_nm(nm)
    q = tuple(filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')))
    async with serve_app(public=True) as srv:
        if srv.port not in pub_ports(): raise RuntimeError(f"Port {srv.port} is not publicly exposed")
        base = '' if srv.port == 8000 else f"https://{pub_domain(srv.port)}"
        url = f'{base}/esm/{nm}' + ('?' if q else '') + '&'.join(q)
        res = await js_eval_a(f'''try {{ return await import({json.dumps(url)}) }} catch (e) {{ return {{__load_mod_error__: `${{e.name}}: ${{e.message}}`}} }}''')
    if err := res['result'].get('__load_mod_error__'): raise RuntimeError(err)
    return res
```

The successful return value is unchanged. A module that throws `new Error("test module failure")` should now promptly raise `RuntimeError: Error: test module failure`.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 57439, "completion_tokens": 849, "total_tokens": 58288, "cached_tokens": 55552, "reasoning_tokens": 516, "cost": 0.009252600000000001}
```


In [ ]:
#| export
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int=0, # Numeric version selecting registered `<nm>_<v>` before falling back to `<nm>`
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> object:
    "Import a registered ES module in the active Solveit browser and return the url and its exports (as JSON)."
    _check_mod_nm(nm)
    async with serve_app(public=True) as srv:
        q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
        base = '' if srv.port == 8000 else f"https://{pub_domain(srv.port)}"
        url = f'{base}/esm/{nm}' + ('?' if q else '') + '&'.join(q)
        res = await js_eval_a(f'''try {{ return await import({json.dumps(url)}) }} catch (e) {{ return {{__load_mod_error__: `${{e.name}}: ${{e.message}}`}} }}''')
    if err := res['result'].get('__load_mod_error__'): raise RuntimeError(err)
    return url, res

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Test the documented failure contract with a fresh module name, since failed ES-module imports are cached by URL in the browser:

```python
reg_mod('test_load_mod_error_2', 'throw new Error("test module failure")')

with expect_fail(RuntimeError, 'Error: test module failure'):
    await load_mod('test_load_mod_error_2')
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 58092, "completion_tokens": 267, "total_tokens": 58359, "cached_tokens": 56576, "reasoning_tokens": 188, "cost": 0.00891405}
```


In [ ]:
reg_mod('test_load_mod_error', 'throw new Error("test module failure")', v=1)

with expect_fail(RuntimeError, 'Error: test module failure'):
    await load_mod('test_load_mod_error', 1)

In [ ]:
reg_mod('test_load_mod', 'globalThis.__test_load_mod = 13; export const answer = 13', v=1)

url,mod = await load_mod('test_load_mod', 1)

test_eq(mod['result']['answer'], 13)
test_eq((await js_eval_a('return globalThis.__test_load_mod'))['result'],13)
url

So, #_ea5b77c4 and the above 2 tests is enough for now, right? And mostly to document and teach more than anything.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. Those three tests are sufficient for `load_mod` at this stage, chiefly as executable documentation:

- `_ea5b77c4`: a base module imports, exposes exports, and executes browser-side code.
- The error test: a failed module import raises promptly with the browser error message rather than timing out.
- The versioned-module test: `load_mod('test_load_mod', 2)` resolves the registered `test_load_mod_2` source and executes it.

They cover the public behavior `load_mod` adds without duplicating tests for URL assembly, validation, server lifecycle, or `js_eval_a`.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 58490, "completion_tokens": 359, "total_tokens": 58849, "cached_tokens": 57600, "reasoning_tokens": 226, "cost": 0.00904275}
```


Are you sycophant-ing me or do you really think in this environment (Solveit/Notebook, literate programming) that's the way to proceed?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I really think this is the right direction for a Solveit/Notebook library, not merely agreement.

The key distinction is between **unit-test completeness** and a **literate, executable specification**. Here, most of `load_mod` is glue over trusted helpers and ordinary Python. Repeating tests for each bit of glue would obscure the small contract the function actually owns.

The three examples state that contract well:

1. Base module import: browser code loads, runs, and its exports come back.
2. Versioned import: version selection reaches the intended module.
3. Failing import: browser errors become immediate, informative Python errors.

They are placed next to the implementation, runnable by a reader, and teach the behaviors a caller needs to know. That is valuable in this environment.

My earlier wording was too categorical: “sufficient” depends on the intended stability and audience. If `load_mod` becomes a widely used public API, I would add a small direct test for its own input contract—perhaps invalid `nm`—*only if* that validation is part of the documented API rather than merely defensive forwarding. But I would still avoid mock-heavy tests of URL concatenation, `serve_app`, or `js_eval_a`.

So: for the current notebook stage, these are the right tests and the right amount of testing.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 58709, "completion_tokens": 524, "total_tokens": 59233, "cached_tokens": 56576, "reasoning_tokens": 248, "cost": 0.00919935}
```


Ok, `nm` is an interesting example. The contract has two parts:

- `nm` is a string; syntax, shape; stated in the signature
- `nm` is a module name both url-safe and javascript-safe; intention, semantics; stated in the docment

That's the end of our resposability as developers of `load_mod`. But, just in case, we make some verifications in `_check_mod`. We really shouldn't, at least in a first version, but still.

In the body of the function we strive for honoring the contract. And that mostly made with correct vanilla python and a well-tested `serve_app`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. That is a sound allocation of responsibility.

`load_mod`’s input contract is declared by:

- the signature: `nm: str`
- the docment: it must be a route-safe module name already registered with `reg_mod`

The function’s job is to use that value consistently to construct and import the module URL. It should not redefine URL syntax, JavaScript-module naming, port allocation, or server lifecycle as new concerns to test internally.

`_check_mod_nm` is a defensive boundary check. Its detailed acceptance/rejection cases belong with `_check_mod_nm`—as they already do—not as tests of `load_mod`. Likewise, `serve_app` owns its lifecycle behavior and should be tested at its own level.

One small refinement: Python’s `nm: str` is a declaration rather than runtime enforcement. But that does not make runtime type-validation mandatory in `load_mod`; it means callers who violate the API may receive whatever clear failure follows. The explicit `_check_mod_nm` improves that failure, but does not expand `load_mod`’s core contract.

So the `load_mod` tests should remain concerned with the behavior it uniquely composes:

- import a browser module and return its exports;
- select a versioned module;
- turn an import failure into an immediate useful Python exception.

That is a compact executable specification, rather than a reconstruction of every dependency’s test suite.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 59172, "completion_tokens": 382, "total_tokens": 59554, "cached_tokens": 57600, "reasoning_tokens": 89, "cost": 0.0091623}
```


But that's not how you or your peers behave regarding this type of programming. When writing tests, the body of the function should be a black box. We shoud test the external behavior according to the explicit contract: signature, docstring and docments. In some cases, test to teach. But try to document well the contract first.

How to achieve that, how to steer the AIs behavior to paths clearly not in their training set and distribution? I don't know, but probably involving a "partial" context: once we develop the function, hide from the context the function body (and any exploration leading to the function body) and asks the assistant to write the tests.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. My earlier rationale still leaked implementation knowledge into the test decision. That is the failure mode: once an assistant has seen the body and its exploratory history, it tends to write *confirmation tests* for the chosen implementation, rather than tests derived independently from the public contract.

The better discipline is:

1. **Write the external contract first**
   - Signature and types
   - Docstring/docments: preconditions, return value, failure modes, lifecycle/ownership
   - A short example only where it teaches an important usage pattern

2. **Implement against that contract**

3. **Test from a partial context**
   - Give the test-writing assistant only the public contract, imports needed to invoke the API, and perhaps adjacent public examples.
   - Hide the function body, helper bodies, mock-heavy exploration, and prior test ideas.
   - Ask it to propose a minimal set of black-box tests: observable inputs, outputs, errors, and effects.

4. **Review tests against the contract**
   - Each assertion must map to a documented promise or a deliberate documentation example.
   - Delete tests that merely preserve incidental implementation choices.
   - Move dependency tests to the dependency that owns the behavior.

For `load_mod`, a contract-only testing prompt might contain:

```python
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int|None=None, # Numeric version selecting registered `<nm>_<v>` before falling back to `<nm>`
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> object:
    "Import a registered ES module in the active Solveit browser and return its exports."
```

Plus `reg_mod`’s public contract and the fact that this runs in a Solveit browser. From that, a good test author should infer only externally visible cases:

- a module’s exports are returned;
- importing executes module top-level code;
- `v` selects a versioned module;
- an import error is prompt and intelligible, if that failure behavior is documented.

It should **not** infer or test:

- `_check_mod_nm` was called;
- `serve_app(public=True)` was used;
- port `8000` got a relative URL;
- query-string formatting;
- `pub_ports()` membership checks;
- the particular JavaScript envelope used to transport errors.

Those may be excellent *implementation-level diagnostic checks during development*, but they are not durable public API tests unless they express a contract.

A practical Solveit pattern would be to put a small contract note immediately above a function, then use a separate, deliberately constrained test-review context or spawned agent whose prompt contains only that note and the callable’s signature. The implementation author can then compare the independently proposed tests with the actual tests. That separation will not be perfect—an LLM still brings priors—but it removes the strongest source of contamination: the code path it has just watched being constructed.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 59655, "completion_tokens": 704, "total_tokens": 60359, "cached_tokens": 57600, "reasoning_tokens": 91, "cost": 0.00947625}
```


###### start-section

In [ ]:
events = []

@asynccontextmanager
async def fake_resolved_server(*args, **kwargs):
    events.append(('enter', kwargs))
    try: yield AD(port=8001)
    finally: events.append(('exit',))

with mock_patch.dict(load_mod.__globals__, serve_app=fake_resolved_server, js_eval_a=fake_js_eval_a,
                     pub_ports=lambda: (8000,8001), pub_domain=lambda port: 'mods.example'):
    # test_eq(await load_mod('demo', port=6551), 'ok')
    test_eq(await load_mod('demo'), 'ok')

test_eq(events, [
    # ('enter', dict(port=6551)),
    ('enter', dict(public=True)),
    ('eval', 'return await import("https://mods.example/esm/demo")'),
    ('exit',),
])

In [ ]:
events = []

@asynccontextmanager
async def fake_serve_app(port=None, **kwargs):
    events.append(('enter', port, kwargs))
    try: yield AD(port=8000)
    finally: events.append(('exit', port))

with mock_patch.dict(load_mod.__globals__, serve_app=fake_serve_app, js_eval_a=fake_js_eval_a,
                     pub_ports=lambda: (8000, 8001), pub_domain=lambda port: 'mods.example'):
    test_eq(await load_mod('demo'), 'ok')

test_eq(events, [
    ('enter', None, dict(public=True)),
    ('eval', 'return await import("/esm/demo")'),
    ('exit', None),
])

Its first event should be `('enter', 8000, {'public': True})`, meaning an existing port-8000 server will be reused and the module import remains same-origin.

The new behavior needs three focused tests: relative default-port import, absolute non-default public-port import, and private-port rejection before browser evaluation.

In [ ]:
events = []

@asynccontextmanager
async def fake_serve_app(port=8000, **kwargs):
    events.append(('enter', port, kwargs))
    try: yield AD(port=port)
    finally: events.append(('exit', port))

async def fake_js_eval_a(src): events.append(('eval', src)); return 'ok'

In [ ]:
with mock_patch.dict(load_mod.__globals__, serve_app=fake_serve_app, js_eval_a=fake_js_eval_a,
                     pub_ports=lambda: (8000,8001), pub_domain=lambda port: 'mods.example'):
    test_eq(await load_mod('demo', v=2), 'ok')
    test_eq(await load_mod('demo', v=2, dname='a b/c', port=8001), 'ok')

test_eq(events, [
    ('enter', 8000, dict(public=True)),
    ('eval', 'return await import("/esm/demo?v=2")'),
    ('exit', 8000),
    ('enter', 8001, dict(public=True)),
    ('eval', 'return await import("https://mods.example/esm/demo?v=2&dlg_name=a%20b%2Fc")'),
    ('exit', 8001),
])

Then reset `events` and pin down the fail-fast contract:

In [ ]:
events = []

with mock_patch.dict(load_mod.__globals__, serve_app=fake_serve_app, js_eval_a=fake_js_eval_a,
                     pub_ports=lambda: (8000,8001)):
    with expect_fail(RuntimeError, 'Port 6551 is not publicly exposed'):
        await load_mod('demo', port=6551)

test_eq(events, [('enter', 6551, {}), ('exit', 6551)])

The last assertion matters: no `eval` event occurs, while context cleanup still runs.

The tests pass. They establish the new transport contract:

- port `8000` imports with a relative URL;
- other public ports import from their assigned absolute domain;
- private ports raise before `js_eval_a` runs;
- `serve_app` still exits and cleans up on that rejection.

###### end-section

In [ ]:
#| export
async def setup_mod(
    nm:str, # Stable base ESM name
    src:str|None=None, # JavaScript source; defaults to the preceding JS message
    versioned:bool=False, # Use a kernel-local counter for `?v=<version>` to bypass browser cache
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> tuple[str,object]:
    "Register stable or versioned JavaScript, import it, and return `(registered_name, browser_result)`."
    _check_mod_nm(nm)
    if src is None: src = await get_js()
    if src is None: raise ValueError(f"Message #{(await read_msg()).id} does not contain Javascript source")
    v = kntr(nm) if versioned else 0
    reg_mod(nm, src, v)
    return await load_mod(nm, v, dname=dname)

`_check_mod_nm`, `get_js`, `reg_mod`, `load_mod` are already tested. What should we test here?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`setup_mod` owns only the composed public workflow:

1. register supplied source;
2. import it in the browser;
3. return the registered name alongside the browser result;
4. when `versioned=True`, make successive revisions independently importable;
5. when `src=None`, use the preceding JavaScript message.

The useful tests are therefore executable examples of those workflows—not mocks of calls to `_check_mod_nm`, `reg_mod`, or `load_mod`.

Start with the ordinary supplied-source case:

```python
nm,res = await setup_mod('test_setup_mod', 'globalThis.__setup_mod = 17; export const answer = 17')

test_eq(nm, 'test_setup_mod')
test_eq(res['result']['answer'], 17)
test_eq((await js_eval_a('return globalThis.__setup_mod'))['result'], 17)
```

Then test the purpose of versioning: two revisions of one stable module name both load their own source, rather than the browser reusing the first import:

```python
nm1,res1 = await setup_mod('test_setup_mod_v', 'export const answer = 1', versioned=True)
nm2,res2 = await setup_mod('test_setup_mod_v', 'export const answer = 2', versioned=True)

test_eq((res1['result']['answer'], res2['result']['answer']), (1,2))
test_eq(nm1.startswith('test_setup_mod_v_'), True)
test_eq(nm2.startswith('test_setup_mod_v_'), True)
test_ne(nm1, nm2)
```

Finally, add one documentation test for the notebook-specific default-source workflow. Place a JavaScript message immediately above it:

```javascript
%%javascript
export const answer = 23
```

```python
nm,res = await setup_mod('test_setup_mod_default')

test_eq(nm, 'test_setup_mod_default')
test_eq(res['result']['answer'], 23)
```

Those three examples cover every public promise in `setup_mod`’s docment. Errors from absent or malformed JS messages, registration semantics, import transport, and name validation remain owned by the already-tested collaborators.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 61385, "completion_tokens": 967, "total_tokens": 62352, "cached_tokens": 3328, "reasoning_tokens": 516, "cost": 0.009933}
```


`setup_mod` owns only the composed public workflow:

1. register supplied source;
2. import it in the browser;
3. return the registered name alongside the browser result;
4. when `versioned=True`, make successive revisions independently importable;
5. when `src=None`, use the preceding JavaScript message.

The useful tests are therefore executable examples of those workflows—not mocks of calls to `_check_mod_nm`, `reg_mod`, or `load_mod`.

Start with the ordinary supplied-source case:

In [ ]:
url,mod = await setup_mod('test_setup_mod', 'window.__setup_mod = 17; export const answer = 17')

test_eq(res['result']['answer'], 17)
test_eq((await js_eval_a('return window.__setup_mod'))['result'], 17)
url

Then test the purpose of versioning: two revisions of one stable module name both load their own source, rather than the browser reusing the first import:

In [ ]:
nm1,res1 = await setup_mod('test_setup_mod', 'export const answer = 1', versioned=True)
nm2,res2 = await setup_mod('test_setup_mod', 'export const answer = 2', versioned=True)

test_eq((res1['result']['answer'], res2['result']['answer']), (1,2))
test_eq(nm1.startswith('test_setup_mod_'), True)
test_eq(nm2.startswith('test_setup_mod_'), True)
test_ne(nm1, nm2)

print(nm1, nm2)

Finally, add one documentation test for the notebook-specific default-source workflow. Place a JavaScript message immediately above it:

In [ ]:
nm,res = await setup_mod('test_setup_mod_msg')

test_eq(nm, 'test_setup_mod_msg')
test_eq(res['result']['answer'], 23)

Those three examples cover every public promise in `setup_mod`’s docment. Errors from absent or malformed JS messages, registration semantics, import transport, and name validation remain owned by the already-tested collaborators.

### ----

In [ ]:
loaded,res = await setup_mod('test', _test_js, True)
test_eq((await hx.eval_a('window._test.hello')), 'world')

empty,res = await setup_mod('empty', '')
test_eq(empty, 'empty')
test_eq(TestClient(mk_app({})).get('/esm/empty').text, '')
empty,res

In [ ]:
reg_nm, res = await setup_mod('test', _test_js, True)
test_eq(reg_nm.split('_')[0], 'test')
test_eq(res.result, {'test': {'hello': 'world'}})
reg_nm, res

In [ ]:
reg_nm, res = await setup_mod('test', _test_js)
test_eq(reg_nm, 'test')
reg_nm, res

In [ ]:
await setup_mod('test2')

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,_,_ = await ensure_server_a(port=port, mods=dict(explicit='export const ok = true'))
test_eq(TestClient(app).get('/esm/explicit').text, 'export const ok = true')
await stop_server_a(port)

In [ ]:
with expect_fail(ValueError, 'does not contain Javascript source'):
    await setup_mod('test3')

The remaining local contract is that `setup_mod` registers the source, then passes `dname` through to `load_mod`. This test pins that down without invoking the browser:

In [ ]:
calls,res = [],[]

async def fake_load_mod(*args, **kwargs):
    calls.append((args, kwargs))
    return 'ok'

with mock_patch.dict(setup_mod.__globals__, load_mod=fake_load_mod):
    res.append(await setup_mod('named', 'export const x = 1', dname='a/b/c'))

test_eq(res, [('named', 'ok')])
test_eq(calls, [(('named', None), dict(dname='a/b/c'))])
test_eq(__mods__['named'], 'export const x = 1')

The examples establish the contract:

1. A module that exports `test` and assigns `window._test` is registered and imported; browser evaluation confirms `window._test.hello == 'world'`.
2. Repeated versioned loads create distinct module names (`test_1`, then `test_2`).
3. A plain `%%js` cell can be loaded from the immediately preceding message; its exported `dname` matches the current dialog path.
4. Calling `setup_mod` when the preceding message is not JavaScript raises a clear `ValueError`.

In short, it is a notebook/dialog development helper for: write JS in a message → register it on the local FastHTML server → dynamically import it into the current browser session.

### Browser ES modules

Register and import JavaScript in the active Solveit browser with:

```python
src = 'export const answer = 42'
nm,res = await setup_mod('editor', src, versioned=True)
```

With `versioned=True`, repeated calls register successive names such as `editor_1` and `editor_2`, and import distinct URLs such as `/esm/editor?v=1`. Use this during iterative development to bypass the browser’s ES-module cache.

The module registry belongs to the current Python kernel, not to an app or server. Registering an existing key replaces its source, and registered entries survive server shutdown. There is currently no public operation for unregistering an entry or clearing the registry.

Registered modules are executable browser code and may be served through the app’s public HTTP endpoint. Register only trusted source; module names are not an authorization boundary.

An unversioned name provides a stable URL. Replacing its registered source changes future HTTP responses, but a browser that has already imported that URL may continue using its cached module.

If given, `dname` is URL-encoded and forwarded as the `dlg_name` query parameter during the browser import. It does not affect module-registry lookup; browser-side code can use it only if it reads that parameter.

`setup_mod` and `load_mod` import through a given or discovered server in the active Solveit browser. The resolved server must listen on a Solveit public port: port `8000` uses a relative import URL, while another public port uses its assigned public domain. A private port raises `RuntimeError` before browser evaluation.

If no live server is found and the requested public port is free, a temporary server is created for the import and stopped afterward. If the port is occupied by a non-discoverable process, setup fails rather than replacing it.

JavaScript can also come from a dialog message whose first line is `%%js` or `%%javascript`:

- `get_js` returns the source without the magic line.
- `setup_mod` without `src` extracts and imports the preceding JavaScript message.
- `add_js_msg` creates linked Python state containing the extracted source.

A missing registry entry is served as a valid diagnostic JavaScript module rather than an HTTP error, so an import may succeed without providing the expected exports.

###### start-section

Let's test programatic import in the browser: `const {...} = await import('/esm/x?v=...')`

In [ ]:
await setup_mod('x', 'export const x=42,y="ok"', True)

In [ ]:
res = await js_eval_a("""const {x,y} = await import('/esm/x?v=1'); return {x,y}""")
test_eq(res['result'], dict(x=42, y='ok'))

How could we use `import('x')` instead of `import('/esm/x?v=1')`

---

A bare specifier such as `import('x')` needs an **import map** in the document where the JavaScript runs. The existing `threejs_hdrs` is the same pattern.

For a fixed module version:

```python
app.hdrs.append(Script(json.dumps(dict(imports=dict(x='/esm/x?v=1'))), type='importmap'))
```

Then, after the browser has loaded that document:

```python
res = await js_eval_a("""const {x,y} = await import('x'); return {x,y}""")
test_eq(res['result'], dict(x=42, y='ok'))
```

For the unversioned, current module, map it to `/esm/x` instead:

```python
app.hdrs.append(Script(json.dumps(dict(imports=dict(x='/esm/x'))), type='importmap'))
```

Native import maps are fixed once module loading has begun. Thus `import('x')` is useful for a stable module identity, but versioned hot-reloads should keep using distinct URLs such as `/esm/x?v=1`, `/esm/x?v=2` (or use a shim that supports mutable import maps).

###### end-section

# pyskill

In [ ]:
import pote.server

list_pyskills()['pote.server']

In [ ]:
doc(pote.server)

In [ ]:
doc(pote.server.ensure_server)

In [ ]:
doc(pote.server.serve_app)

In [ ]:
doc(pote.server.setup_mod)

# export -

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()